# Практика · Класифікація текстів

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.md](homework.md) ·
> Тест: [quiz.html](quiz.html)

Тут рахуються **всі** числа, які називає лекція, і в тому самому порядку.

Що зробимо:

1. Зберемо корпус українських перекладів і поставимо мітку з **англійського** оригіналу.
2. Перевіримо, що мітка чесна: що модель не бачить того, з чого її зроблено.
3. Порахуємо дублікати в корпусі й заміряємо, скільки коштує витік через них.
4. Навчимо чотири моделі на трьох зернах і зберемо головну таблицю теми.
5. Заміряємо, наскільки хибне припущення наївного Баєса про незалежність слів.
6. Доведемо `assert`-ом, що `ComplementNB` на двох класах — це `MultinomialNB`
   без апріорної ймовірності, і нічого більше.
7. Зʼясуємо, чому `class_weight='balanced'` не лише кращий, а й швидший.
8. Витягнемо ваги логістичної регресії й подивимось, чого саме вона навчилась.

> ⏱ Заміряно: близько **двох хвилин** на чотирьох ядрах без відеокарти
> (два прогони поспіль дали 137 і 120 секунд). Найдовше йде розгортка дисбалансу
> (двадцять одна векторизація) і крива навчання.

## 1 · Середовище

Перша клітинка друкує версії. Якщо в тебе інші — числа можуть трохи поїхати,
і краще знати про це одразу, а не наприкінці.

Вона ж просить числові бібліотеки рахувати в **один потік**. Це не оптимізація,
а умова того, щоб замір часу взагалі щось означав: на машині, де вже щось
рахується, чотири потоки більшу частину часу чекають одне на одного — і це
очікування записується в процесорний час. Заміряно на цьому корпусі: одне й те
саме навчання логістичної регресії коштує **0.24 с** процесорного часу в один
потік і **понад 10 с** у чотири. Результат при цьому побітово той самий.

In [ ]:
import os
# Просимо numpy і scipy не розпаралелювати обчислення.
# Ставити треба ДО імпорту numpy — пізніше вже не подіє.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import sys, re, math, glob, gettext, time, json, collections
import numpy as np
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from sklearn.base import clone

print("Python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("sklearn ", sklearn.__version__)
print("потоків :", os.environ["OMP_NUM_THREADS"])

## 2 · Корпус: ті самі переклади інтерфейсів

Корпус той самий, що в усьому блоці 1: пари «англійський оригінал → український
переклад» із `.mo`-файлів української локалі. Ми беремо звідти **обидва** боки, і
далі побачимо навіщо.

Якщо української локалі на машині немає, вмикається вбудований запасний корпус —
маленький, але достатній, щоб зошит виконався до кінця. Він друкує гучне
попередження: числа на ньому будуть **інші**, і посилатися на них не можна.

In [ ]:
def load_system_corpus():
    """Читаємо .mo-файли української локалі.
    Повертаємо трійки (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                docs.append((program, source, target))
    return docs


FALLBACK_PAIRS = [
[
"Get information of domain's mounted filesystems.",
"Отримати інформацію щодо змонтованих файлових систем домену."
],
[
"reconfigure all registered enlistments",
"переналаштувати всі зареєстровані кореневі директорії проекту"
],
[
"Nested display hints in S-expression",
"Вкладені настанови щодо показу у S-виразі"
],
[
"Ignore invalid packets from i_nactive ports",
"Ігнорувати некоректні пакети з _неактивних портів"
],
[
"add given FILE to the archive (useful if its name starts with a dash)",
"долучити ФАЙЛ до архіву (корисне, якщо його назва починається з -)"
],
[
"Wipe status bar upon next keystroke",
"Витерти смужку стану після натискання клавіші"
],
[
"<b>Too much inset</b>, the result is empty.",
"<b>Надто багато втягувань</b>, результат порожній."
],
[
"Digital zoom not used",
"Цифровий трансфокатор не використовувався"
],
[
"relocation cannot be done when using -mrelocatable",
"пересування не можна виконувати, якщо використовується -mrelocatable"
],
[
"Results for applications contained in this list will be displayed when searching.",
"Результати для програм з цього списку будуть показані при пошуку."
],
[
"Byte address required. - must be even.",
"Потрібна байтова адреса. - має бути парним."
],
[
"ICU is not supported in this build",
"ICU не підтримується в цій збірці"
],
[
"Warn about casts which discard qualifiers.",
"Попереджувати про приведення, які відкидають кваліфікатори."
],
[
"Regexp I-search: ",
"I-пошук за формальним виразом: "
],
[
"Conform to the ISO Fortran 2003 standard.",
"Відповідати стандарту ISO Fortran 2003."
],
[
"Cannot enable more streams from module '{}' at the same time",
"Не можна вмикати додаткові потоки з модуля «{}» одночасно"
],
[
"usage: ifdown-routes <net-device> [<nickname>]",
"користування: ifdown-routes <пристрій-мережі> [<псевдонім>]"
],
[
"Unable to clear thread local variable",
"Не вдалося спорожнити локальну змінну потоку виконання"
],
[
"Enter string for PDF Version, e.g. 1.4 or 1.5",
"Введіть рядок версії PDF. Приклад: 1.4 або 1.5"
],
[
"types may not be defined in conditions",
"типи не можуть бути визначені в умовах"
],
[
"Failed to read LUKS2 requirements.",
"Не вдалося прочитати вимоги LUKS2."
],
[
"Dynamic Range Wide Mode",
"Режим широкого динамічного діапазону"
],
[
"Enter PUK to unlock your SIM card",
"Введіть PUK-код для розблокування вашої SIM-картки"
],
[
"MATCH PARTIAL not yet implemented",
"Вираз MATCH PARTIAL все ще не реалізований"
],
[
"Illegal Parameter Count for ConstructObject Method",
"Неправильна кількість параметрів у методі ConstructObject"
],
[
"call is considered never executed and code size would grow",
"виклик вважається ніколи не виконуваним і розмір коду збільшиться"
],
[
"Announce block_quotes during navigation",
"Оголошувати блоки _цитат під час навігації"
],
[
"Just Type to Search",
"Просто наберіть щось на клавіатурі для пошуку"
],
[
"Graphite loop optimizations cannot be used (isl is not available).",
"Оптимізації петель Graphite не можуть бути використані (isl недоступний)."
],
[
"Russian (Kazakhstan, with Kazakh)",
"Російська (Казахстан, з казахською)"
],
[
"Chunk-based file formats",
"Файлові формати на основі шматків"
],
[
"Support code generation of sahf instruction in 64bit x86-64 code.",
"Підтримка генерації коду для інструкції sahf в 64-бітовому x86-64 коді."
],
[
"Share this screenshot with the requesting application?",
"Оприлюднити цей знімок екрана за допомогою відповідної програми?"
],
[
"Error in reading image DIB.",
"Помилка під час читання картинки DIB."
],
[
"Diagonal fractions. OpenType table: 'frac'",
"Діагональні дроби. Таблиця OpenType: «frac»"
],
[
"Right Ctrl as Right Alt",
"Права клавіша Ctrl працює як права клавіша Alt"
],
[
"opcode not valid for this cpu variant",
"код операції є нечинним для цього варіанта процесора"
],
[
"Failed to set splash mode",
"Не вдалося встановити режим вітання"
],
[
"[0-9]H labels may not appear alone on a line",
"мітки [0-9]H не можуть бути єдиними даними у рядку"
],
[
"The cell renderer represented by this accessible",
"Обробник комірки представлений цими доступностями"
],
[
"Show raw dump of the PCI configuration space.",
"Вивести необроблений дамп простору налаштовування PCI."
],
[
"single-precision hard float, ",
"апаратна рухома крапка з одинарною точністю, "
],
[
"has from/src but the prefix-length is zero",
"містить from/src, але значення довжини префікса (prefix-length) є нульовим"
],
[
"Model word must be in the dictionary. Press any key!",
"Модельне слово має бути у словнику. Натисніть будь-яку клавішу!"
],
[
"-mcpu=m16c\tCompile code for M16C variants.",
"-mcpu=m16c\tКомпілювати код для варіантів M16C."
],
[
"Use mobile data when roaming",
"Використовувати мобільний доступ до даних при роумінгу"
],
[
"(PowerPC only) Align PLT call stubs to fit cache lines",
"(лише PowerPC) Вирівняти фіктивні виклики PLT за рядками кешу"
],
[
"Reload NetworkManager configuration",
"Перезавантажити налаштування NetworkManager"
],
[
"instruction does not allow shifted index",
"у інструкції заборонено індексування зі зсувом"
],
[
"No running virtual machine found",
"Не знайдено запущених віртуальних машин"
],
[
"If you want to set more than one attribute, you must separate this with a space, and only with a space.",
"Якщо ви бажаєте встановити декілька атрибутів, вам слід відокремити їх у списку пропуском і лише пропуском."
],
[
"Missing 'runtime.powerState' property",
"Не вказано властивості «runtime.powerState»"
],
[
"Failed to set symmetric key for decryption.",
"Не вдалося встановити симетричний ключ для розшифрування."
],
[
"-umbrella <framework>\tThe specified framework will be re-exported.",
"-umbrella <фреймворк>\tВказаний фреймворк буде повторно експортований."
],
[
"WB Info A100",
"Відомості про баланс білого для A100"
],
[
"entering main parallel loop",
"введення головного паралельного циклу"
],
[
"Memory exhausted",
"Вичерпано доступний об’єм пам’яті"
],
[
"Do not perform device safety checks",
"Не виконувати перевірок безпечності для пристрою"
],
[
"Whether to remove old files from the trash automatically",
"Чи вилучати автоматично старі файли зі смітника"
],
[
"intermediate certificate not yet valid",
"проміжний сертифікат ще не набув чинності"
],
[
"Base DN for IP networks lookups",
"Базова назва домену для пошуків IP-мереж"
],
[
"Use packed stack layout.",
"Використовуйте упаковану структуру стеку."
],
[
"Warning: all data on the volume will be lost",
"Попередження: усі дані тому буде втрачено"
],
[
"number in parallel must be nonzero",
"кількість паралельних впорядкувань має бути ненульовою"
],
[
"Select the touchpad scroll method",
"Виберіть для сенсорного пристрою спосіб прокручування"
],
[
"Session names are not allowed to contain “/” characters",
"Назвам сеансів не дозволено містити символи «/»"
],
[
"COPY force null available only in CSV mode",
"Параметр force null для COPY можна використати тільки в режимі CSV"
],
[
"Error in relocation handling",
"Помилка під час обробки пересування"
],
[
"_Embed your dictionary in the system dictionary",
"В_будувати ваш словник до загальносистемного словника"
],
[
"multi-pack-index reverse-index chunk is the wrong size",
"multi-pack-index reverse-index шматок має невірний розмір"
],
[
"Read a firmware blob from a device",
"Прочитати бінарну мікропрограму з пристрою"
],
[
"Database for table does not exist",
"Бази даних для таблиці не існує"
],
[
"Please specify Input Chars",
"Будь ласка, вкажіть символи введення"
],
[
"Convert an AppData file to NEWS format",
"Перетворити файл AppData у формат NEWS"
],
[
"Toggle lamp of flatbed",
"Перемкнути лампу планшетного сканера"
],
[
"Generate big-endian code.",
"Генерувати код у великому порядку байтів."
],
[
"The style on which this style is based.",
"Стиль, на якому засновано цей стиль."
],
[
"protocol misses the family attribute",
"у протоколі пропущено атрибут сімейства"
],
[
"postfix operators are not supported",
"постфіксні оператори не підтримуються"
],
[
"Build pack index file for an existing packed archive",
"Побудувати індексний файл пакунка для існуючого запакованого архіву"
],
[
"stack size must not be greater than 64k",
"розмір стеку не повинен перевищувати 64к"
],
[
"timeout writing to relay",
"перевищено час очікування на запис до ретранслятора"
],
[
"calls to overloaded operators cannot appear in a constant-expression",
"виклики перевантажених операторів не можуть зʼявлятися в константному виразі"
],
[
"Creating AppInfo from id not supported on non unix operating systems",
"Створення AppInfo з ідентифікатора не підтримується на системах відмінних від UNIX"
],
[
"Disable batching of geometry in the Cogl Journal.",
"Вимкнути пакування геометрії в журналі Cogl."
],
[
"amount of data to upload",
"об'єм даних, які слід вивантажити"
],
[
"Whether to filter rules by hostname, IP addresses and network",
"Визначає, чи слід фільтрувати правила за назвами вузлів, IP-адресами та мережами"
],
[
"Reorder basic blocks and partition into hot and cold sections.",
"Переставити базові блоки та розбити на гарячі та холодні секції."
],
[
"container format the data is stored in",
"формат контейнера, у якому зберігаються дані"
],
[
"Plains Indian Sign Language",
"рівнинна індіанська мова жестів"
],
[
"git range-diff [<options>] <base> <old-tip> <new-tip>",
"git range-diff [<опції>] <база> <стара-верхівка> <нова-верхівка>"
],
[
"Which Audioscrobbler services do you wish to use?",
"Які служби Audioscrobbler ви бажаєте використовувати?"
],
[
"Size of the node number labels (20px, 12pt...)",
"Розмір міток номерів вузлів (20px, 12pt...)"
],
[
"qemu does not support more than one entry to Type 2 in SMBIOS table",
"у qemu не передбачено підтримку понад одного запису типу 2 у таблиці SMBIOS"
],
[
"use <name> instead of the real target",
"використовувати <назву> замість реальної цілі"
],
[
"If TRUE the IFF_VNET_HDR the tunnel packets will include a virtio network header.",
"Якщо IFF_VNET_HDR має значення TRUE, тунельовані пакети включатимуть заголовок мережі virtio."
],
[
"JIS90 forms. OpenType table: 'jp90'.",
"Форми JIS90. Таблиця OpenType: «jp90»."
],
[
"Add Login Mapping. User Mapping will be created when Update is applied.",
"Додати прив’язку реєстраційного запису. Прив’язку буде створено під час застосування оновлення."
],
[
"Connections from inside daemon must be direct",
"З'єднання із внутрішньої фонової служби мають бути безпосередніми"
],
[
"Enter comma-separated string for attribute name, attribute value",
"Введіть рядок відокремлених комами записів «назва атрибуту, значення атрибуту»"
],
[
"Unsupported capacity-to-allocation relation",
"Некоректне співвідношення між місткістю та потрібним об'ємом пам'яті"
],
[
"Decode low-level symbol names into source code names",
"Визначати за низькорівневими назвами символів назви у початковому коді"
],
[
"Use the hardware barrel shifter instead of emulation.",
"Використовуйте револьверний зсувний регістр замість емуляції."
],
[
"The openDB() function cannot open rpm database.",
"Функція openDB() не може відкрити базу даних rpm."
],
[
"missing 'usage' attribute for RAM filesystem",
"не вистачає атрибута «usage» для файлової системи у оперативній пам'яті"
],
[
"Enables the site-specific compatibility workarounds",
"Увімкнути специфічну для сайта зміну параметрів перегляду"
],
[
"(Obsolete) LD_PREBIND is no longer supported.",
"(Застаріло) LD_PREBIND більше не підтримується."
],
[
"Use poppler when importing via commandline",
"Використовувати poppler при імпортуванні за допомогою командного рядка"
],
[
"<b>Redirected device</b>",
"<b>Переспрямований пристрій</b>"
],
[
"Use MIPS-DSP instructions.",
"Використовувати інструкції MIPS-DSP."
],
[
"Unable to change lifecycle action.",
"Не можна змінювати дію життєвого циклу."
],
[
"Navigate to the previous page of effects",
"Перейти до попередньої сторінки ефектів"
],
[
"Cannot register service",
"Не вдалося зареєструвати службу"
],
[
"Relocatable values require at least WORD storage",
"Для придатних до пересування значень потрібне принаймні сховище WORD"
],
[
"High Fidelity Capture (A2DP Source)",
"Високоточне захоплення (джерело A2DP)"
],
[
"pool does not support pool deletion",
"резервним сховищем не передбачено підтримки вилучення буфера"
],
[
"Do not skip directories on different file systems. Ignored if DIRECTORY is not specified.",
"Не пропускати каталоги у інших файлових системах. Буде проігноровано, якщо не вказано DIRECTORY."
],
[
"cannot truncate temporary tables of other sessions",
"тимчасові таблиці інших сеансів не можна скоротити"
],
[
"Failed to access lock file.",
"Не вдалося отримати доступ до файла замка."
],
[
"The certificate has no peers",
"У сертифікаті не вказано вузлів сертифікації"
],
[
"QEMU NBD server does not support TLS transport",
"на сервері NBD QEMU не передбачено підтримки передавання даних TLS"
],
[
"Strength of matte lamination pattern (-5 through +5)",
"Інтенсивність узору матової ламінації (від -5 до +5)"
],
[
"A tag used to record fractions of seconds for the <DateTimeDigitized> tag.",
"Ця мітка використовується для записування часток секунд для мітки <DateTimeDigitized>."
],
[
"JavaScript can access clipboard",
"Скрипти JavaScript можуть отримувати доступ до буфера обміну даними"
],
[
"It would be useful to add a long description to this font to present it better to users.",
"Варто додати повний опис до цього шрифту, щоб краще представити його користувачам."
],
[
"Error storing transaction: {}",
"Помилка при збереженні операції: {}"
],
[
"Sidebar image for the assistant page",
"Зображення збоку сторінки помічника"
],
[
"Invalid destination register for this instruction, use 'tfr'.",
"Некоректний регістр призначення для цієї інструкції, скористайтеся «tfr»."
],
[
"Set the default real kind to an 8 byte wide type.",
"Встановити тип за замовчуванням для дійсних чисел як тип шириною 8 байт."
],
[
"Select <b>object(s)</b> to convert to pattern.",
"Позначте <b>об'єкт(и)</b> для перетворення на візерунок."
],
[
"Support only one IPv4 default gateway",
"Підтримка лише одного типового шлюзу IPv4"
],
[
"The document is automatically reloaded on file change.",
"Документ автоматично перезавантажуватиметься за зміни файла"
],
[
"Open compressed exchange files saved in Corel DRAW",
"Відкрити файли compressed exchange, збережені за допомогою Corel DRAW"
],
[
"Download Folder Read Access",
"Доступ до читання теки отриманих з інтернету даних"
],
[
"Could not open audio device for recording. Device is being used by another application.",
"Не вдалося відкрити пристрій для запису. Пристрій використовується сторонньою програмою."
],
[
"Link contained '//', converted to absolute link.",
"Посилання, що містило '//', перетворено на абсолютне посилання."
],
[
"filter through PROG (must accept -d)",
"фільтрувати через ПРОГРАМУ (має приймати B<-d>)"
],
[
"Could not delete all interface  mappings",
"Не вдалося вилучити всі прив’язки інтерфейсів"
],
[
"no registered fragment for literal",
"немає зареєстрованого фрагмента для літерала"
],
[
"Copyright (max. 20 characters)",
"Авторське право (макс. 20 символів)"
],
[
"The component is an addon, but no `extends` tag was specified.",
"Компонент є додатком, але не вказано теґ «extends»."
],
[
"Run an arbitrary qemu guest agent command; use at your own risk",
"Виконати довільну команду агента гостьової системи qemu; користуйтеся обережно"
],
[
"Use CD-ROM as root.",
"Використовувати компакт-диск як кореневу теку."
],
[
"'<local>' element missing for 'udp' socket interface",
"не вказано елемента «<local>» для інтерфейсу сокета «udp»"
],
[
"Adding exports to output file",
"Експортні дані додаються у файл для виводу"
],
[
"read exclude patterns from <file>",
"читати шаблони виключення з <файлу>"
],
[
"pretty-print any JSON output",
"форматоване виведення для усіх виведених даних JSON"
],
[
"copy to a FD passed disk source is not yet supported",
"підтримки копіювання переданого за допомогою дескриптора файла джерела диска ще не передбачено"
],
[
"only lower 16-bits of first operand are used",
"використано лише останні 16 бітів першого операнда"
],
[
"Could not get SPARQL connection",
"Не вдалося отримати з'єднання SPARQL"
],
[
"Report various link-time optimization statistics.",
"Повідомляти різні статистичні дані про оптимізацію на етапі звʼязування."
],
[
"HTTP/Socks proxy username passed to OpenVPN when prompted for it.",
"Ім'я користувача HTTP/Socks-проксі, яке буде передано до OpenVPN, якщо надійде запит щодо імені."
],
[
"Realm does not support membership using a one time password",
"У області не передбачено членства на основі одноразового пароля"
],
[
"Both arguments must be substitution symbols",
"Обидва аргументи мають бути символами заміни"
],
[
"Authentication is required to suspend the system while an application is inhibiting this.",
"Для того, щоб призупинити систему, коли програми намагаються перешкодити цьому, слід пройти розпізнавання."
],
[
"Cool White Fluorescent (W 3900 - 4500K)",
"Флуоресцентна лампа з холодним білим світлом (W3900 - 4500K)"
],
[
"Cannot use virtio serial for parallel/serial devices",
"Не можна використовувати послідовний порт virtio для пристроїв з паралельним/послідовним зв'язком"
],
[
"Realm does not support automatic membership",
"У області не передбачено автоматичного членства"
],
[
"The PrinterOption backing this widget",
"PrinterOption для цього віджета"
],
[
"cannot open EBL backend",
"не вдалося відкрити канал сервера EBL"
],
[
"Non standard keyslots alignment, manual repair required.",
"Нестандартне вирівнювання слотів ключів, слід виправити дані вручну."
],
[
"The previous message repeated once.",
"Попереднє повідомлення повторено один раз."
],
[
"Message was restarted too many times",
"Повідомлення було перезапущено надто велику кількість разів"
],
[
"Warn about maybe uninitialized automatic variables.",
"Попереджувати про можливо неініціалізовані автоматичні змінні."
],
[
"Keyboard shortcut to switch to the last tab",
"Комбінація клавіш для перемикання на останню вкладку"
],
[
"core.img version mismatch",
"невідповідність версій core.img"
],
[
"failed to connect to the hypervisor",
"помилка з'єднання з гіпервізором"
],
[
"Peer has terminated the connection",
"З’єднання розірвано рівноправним вузлом"
],
[
"name of existing snapshot to make current",
"назва вже створеного знімка, який слід зробити поточним"
],
[
"The number of elements for which hash table verification is done for each searched element.",
"Кількість елементів, для яких виконується перевірка хеш-таблиці для кожного шуканого елемента."
],
[
"Enable the use of blank format items in format strings.",
"Увімкнути використання порожніх елементів формату в рядках форматування."
],
[
"--resolve-git-dir requires an argument",
"--resolve-git-dir потребує аргументу"
],
[
"cannot parse CLDR rule",
"не вдалося обробити правило CLDR"
],
[
"Can't capture new images. Unknown error",
"Захоплення нових зображень неможливе. Невідома помилка."
],
[
"missing interface information",
"не визначає даних щодо інтерфейсу"
],
[
"either -fopenacc or -fopenmp must be set",
"необхідно встановити або -fopenacc, або -fopenmp"
],
[
"Export layers as separate top-level resources",
"Експортувати шари як окремі ресурси верхнього рівня"
],
[
"If on, connector attachment points will not be shown for text objects",
"Якщо вибрано,  для текстових об'єктів не буде показано точки приєднання з'єднувальних ліній"
],
[
"-fcheck=[...]\tSpecify which runtime checks are to be performed.",
"-fcheck=[...]\tВказує, які перевірки часу виконання мають бути виконані."
],
[
"Directories to show in the browse interface, none by default",
"Каталоги, які показуються в огляді, типово нічого не показується"
],
[
"list network routes",
"показати список мережевих маршрутів"
],
[
"Could not list named graphs",
"Не вдалося створити список іменованих графів"
],
[
"Enable double load/store instructions for ARC HS.",
"Увімкнути подвійні інструкції завантаження/збереження для ARC HS."
],
[
"turns on runtime checking for functions which finish without executing a RETURN statement",
"увімкнути перевірку часу виконання для функцій, які завершуються без виконання оператора RETURN"
],
[
"math symbol\u0004square original of or not equal to",
"квадратний оригінал і не дорівнює"
],
[
"Enable DEC-style STATIC and AUTOMATIC attributes.",
"Увімкнути атрибути STATIC та AUTOMATIC у стилі DEC."
],
[
"Can't set value on section node",
"Не можна встановлювати значення на вузлі розділу"
],
[
"RAS image has unknown type",
"RAS-зображення має невідомий тип"
],
[
"USB is disabled for this domain, but USB devices are present in the domain XML",
"USB для цього домену вимкнено, але у XML домену є пристрої USB"
],
[
"Failed to load RGB data from TIFF file",
"Не вдалося завантажити дані RGB з файлу формату TIFF"
],
[
"Spanish (Latin American, no dead keys)",
"Іспанська (латиноамериканська, без сліпих клавіш)"
],
[
"Use the same collation as in the template database, or use template0 as template.",
"Використайте те ж саме правило сортування, що і в шаблоні бази даних, або виберіть template0 в якості шаблона."
],
[
"Paint a translucent random colour over each newly drawn tile",
"Малювати прозорим випадковим кольором усі наново намальовані плитки"
],
[
"restore the value of a run-time parameter to the default value",
"відновити початкове значення параметру виконання"
],
[
"Toggle snapping to tangential lines",
"Увімкнути або вимкнути прилипання до дотичних ліній"
],
[
"Set a custom snooze time for",
"Встановити спеціальний час затримки для"
],
[
"bad insn to frv_print_operand, 'F' modifier:",
"неправильна інструкція для frv_print_operand, модифікатор «F»:"
],
[
"The 256 bit private key in base64 encoding",
"256-бітовий закритий ключ у кодуванні base64"
],
[
"@itemx must follow @item",
"@itemx слід використовувати після @item"
],
[
"invalid data in extended tar header",
"некоректні дані у розширеному заголовку tar"
],
[
"Register name expected",
"Мало бути вказано назву регістра"
],
[
"Automatically locking the screen prevents others from accessing the computer while you're away.",
"Автоматичне блокування екрана запобігає доступу інших користувачів до комп'ютера, поки вас немає."
],
[
"Location was added in OpenStreetMap",
"Місце було додано в OpenStreetMap"
],
[
"backup TLS directory not configured",
"каталог резервного копіювання TLS не налаштовано"
],
[
"Help: Transition application file Page",
"Довідка: сторінка файлів програм"
],
[
"Chunk size of omp schedule for loops parallelized by parloops.",
"Розмір порції розкладу omp для циклів, які паралелізуються за допомогою parloops."
],
[
"Provides support for firmware upgrades",
"Надає підтримку оновлення мікропрограм"
],
[
"Sudo rule runasuser attribute",
"Атрибут користувача, від імені якого виконуватиметься запуск, правила sudo"
],
[
"Do not prompt to fix security issues",
"Не питати про дії щодо проблем із захистом"
],
[
"list valid event types",
"показати список коректних типів подій"
],
[
"Could not load TIFF data",
"Не вдалося завантажити дані TIFF"
],
[
"HTML Help Project (*.hhp)|*.hhp|",
"Проект довідки HTML (*.hhp)|*.hhp|"
],
[
"Unable to read net device config on this platform",
"Неможливо прочитати налаштування мережевого пристрою на цій платформі"
],
[
"Read copy count value",
"Прочитати значення кількості копій"
],
[
"Identifies city of object data origin according to guidelines established by the provider.",
"Вказує місто, на яке посилаються дані об’єкту, у відповідності з методикою, визначеною постачальником."
],
[
"Hit EOF while fetching headers",
"Надіслано EOF під час отримання заголовків"
],
[
"@setfilename after the first element",
"@setfilename після першого елемента"
],
[
"vsock device is not supported with this QEMU binary",
"підтримки пристроїв vsock у цьому виконуваному файлі QEMU не передбачено"
],
[
"Whether to prefer the icon over text",
"Чи слід надавати перевагу піктограмам замість тексту"
],
[
"unexpected placeholder in constrained result type",
"неочікуваний заповнювач у обмеженому типі результату"
],
[
"Set the max size of data eligible for the TDA area.",
"Встановити максимальний розмір даних, придатних для області TDA."
],
[
"The environment is too large for exec().",
"Оточення надто велике для виконання."
],
[
"variables - Names and meanings of some shell variables",
"variables - назви та значення деяких змінних оболонки"
],
[
"Warn about underflow of numerical constant expressions.",
"Попереджати про недостатність числових константних виразів."
],
[
"only decorate refs that match <pattern>",
"оздоблювати лише посилання, що відповідають <шаблону>"
],
[
"Read 32-bit value from ADDR.",
"Прочитати 32-бітове значення за АДРЕСОЮ."
],
[
"Map Unicode to Symbol font",
"Пов'язати Unicode зі шрифтом Symbol"
],
[
"Get domain launch security info",
"Отримати відомості щодо безпеки запуску домену"
],
[
"identifier expected after pre-increment or pre-decrement",
"пре-інкремент чи пре-декремент потребують ідентифікатор"
],
[
"Check to suppress hyphenation.",
"Позначте, щоб придушити перенесення слів."
],
[
"You must be authenticated to insert or move documents and folders.",
"Щоб вставляти або переміщати файли й теки, слід пройти розпізнавання."
],
[
"Format to use: new (default), old, or compat",
"Формат, яким слід скористатися: new (типовий), old або compat"
],
[
"Used for generating code for some older kernel revisions.",
"Використовується для генерації коду для деяких старіших версій ядра."
],
[
"Failed to create thread to handle firewall reload/restart",
"Не вдалося створити гілку обробки для роботи із перезавантаженням або перезапуском брандмауера"
],
[
"only one type should be specified for operand",
"для операнда мало бути вказано лише один тип"
],
[
"member functions are implicitly friends of their class",
"члени-функції є неявними друзями свого класу"
],
[
"Load extra configuration items",
"Завантажити додаткові пункти налаштувань"
],
[
"relocation for non-REL psect",
"пересування для psect, який не є REL"
],
[
"stopped, with saved guests",
"зупинено зі збереженням гостьових систем"
],
[
"Width, in pixels, of the focus indicator line",
"Ширина, в точках, лінії індикатора фокуса"
],
[
"Invalid DIE in debug info; failed to reduce debug info",
"Некоректне DIE у діагностичних даних; не вдалося стиснути діагностичні дані"
],
[
"invalid node cpu cores value",
"некоректне значення кількості ядер процесора вузла"
],
[
"left operand of comma operator cannot resolve address of overloaded function",
"лівий операнд оператора коми не може вирішити адресу перевантаженої функції"
],
[
"Show a long list with more detailed information.",
"Показати довгий список з додатковими відомостями."
],
[
"dynamic index information not available",
"дані динамічного покажчика недоступні"
],
[
"Installing group/module packages",
"Встановлюємо пакунки групи або модуля"
],
[
"invalid Rt register number in 64-byte load/store",
"некоректний номер регістра Rt у 64-байтовому load/store"
],
[
"All small-caps (uppercase and lowercase). OpenType tables: 'c2sc' and 'smcp'",
"Усі малі прописні (верхнього і нижнього регістру). Таблиці OpenType: «c2sc» і «smcp»"
],
[
"Spacing between step buttons and thumb",
"Відстань між кнопками переміщення і вказівником"
],
[
"(ARM only) Set R_ARM_TARGET2 relocation type",
"(лише ARM) встановити тип пересування R_ARM_TARGET2"
],
[
"Set the height of the displayed window",
"Встановити висоту показуваного вікна"
],
[
"Create a desktop-entry file from a metainfo file.",
"Створити файл запису desktop з файла метаінформації."
],
[
"your account has expired",
"термін дії вашого облікового запису вичерпано"
],
[
"Reverses the second path order",
"Обернути напрямок другого контуру"
],
[
"Callbacks missing for ET_REL file",
"Немає зворотних викликів для файла ET_REL"
],
[
"typedef may not be a function definition",
"typedef не може бути визначенням функції"
],
[
"Database is busy",
"Базу даних зайнято виконанням завдання"
],
[
"Credentials required in order to print",
"Реєстраційні дані, які потрібні для друку"
],
[
"Polish (programmer Dvorak)",
"Польська (програмістський Дворак)"
],
[
"advisories about any versions of installed packages",
"поради щодо будь-яких версій встановлених пакунків"
],
[
"Please check whether the file is accessible.",
"Будь ласка, перевірте чи доступний файл."
],
[
"Static binding to lower the volume.",
"Статична прив'язка для зменшення гучності."
],
[
"invalid proto_version",
"неприпустиме значення proto_version"
],
[
"If TRUE, applications should not make sound.",
"Якщо має значення «true», програмам буде заборонено відтворювати звукові дані."
],
[
"|URL|redirect all HTTP requests to URL",
"|URL|переспрямувати всі запити HTTP на вказану адресу"
],
[
"OpenType layout\u0004Vertical Alternates and Rotation",
"Вертикальні альтернативи і обертання"
],
[
"The partition is being detached concurrently or has an unfinished detach.",
"Розділ відключається одночасно або має незакінчене відключення."
],
[
"shortcut window\u0004Cancel current changes for contact",
"Скасувати поточні зміни до запису контакту"
],
[
"expected expression-list or type-id",
"очікується список виразів або ідентифікатор типу"
],
[
"You should give exactly one directory name",
"Вам слід вказати точно одну назву каталогу"
],
[
"Reset your display to the factory defaults.",
"Скинути параметри екрана до типових заводських."
],
[
"<b>No texts-on-paths</b> in the selection.",
"У позначеному <b>немає тексту за контуром</b>."
],
[
"use merging strategies to rebase",
"використовувати стратегії злиття для перебазування"
],
[
"The autovideosink element is missing.",
"Не вказано елемента autovideosink."
],
[
"Idle time before automatic disconnection of a client",
"Проміжок бездіяльності до автоматичного від’єднання клієнтської частини"
],
[
"Authentication is required to print this document",
"Для друку документу потрібно пройти автентифікацію"
],
[
"Error: Option '--pretty' is specified the second time.",
"Помилка: параметр «--pretty» вказано двічі."
],
[
"Allow (or do not allow) gcc to use the LOOP instruction.",
"Дозволити (або не дозволяти) gcc використовувати інструкцію LOOP."
],
[
"must be superuser to alter superuser roles or change superuser attribute",
"для зміни ролей суперкористувача або зміни атрибуту суперкористувача потрібно бути суперкористувачем"
],
[
"pci-root and pcie-root controllers should have index 0",
"Кореневі контролери PCI та PCIE повинні мати індекс 0"
],
[
"Left/right mirror image",
"Віддзеркалити зображення горизонтально"
],
[
"Protocol wrong type for socket",
"Помилковий тип протоколу сокета"
],
[
"Could not read container config",
"Помилка при читанні конфігурації контейнера"
],
[
"Can't specify more than one of VNC, SDL, --graphics or --nographics",
"Можна використовувати лише один з параметрів VNC, SDL, --graphics або --nographics"
],
[
"Transformed JPEG has zero width or height.",
"Перетворене зображення формату JPEG має нульову ширину чи висоту."
],
[
"rule WHERE condition cannot contain references to other relations",
"в умовах WHERE правила не можуть містити посилання на інші зв'язки"
],
[
"AMO insns require rd != base && rd != rt when rd isn't $r0",
"Інструкції AMO потребують rd != base && rd != rt, якщо rd не збігається з $r0"
],
[
"missing IFLA_VF_INFO in netlink response",
"у відповіді netlink не вистачає IFLA_VF_INFO"
],
[
"Homepage for this media (i.e. artist or movie homepage)",
"Домашня сторінка цього носія даних (домашня сторінка виконавця або фільму)"
],
[
"Linking user relation files",
"Приєднання файлів користувацьких відношень"
],
[
"Provide authentication form responses",
"Надати відповіді для форми розпізнавання"
],
[
"return to real time",
"повернення до до режиму реального часу"
],
[
"Add direction randomness by moving 'bottom' half-turns tangentially to the boundary.",
"Додати випадковість напрямку пересуванням «нижніх» напіввигинів паралельно до межі."
],
[
"How much can given compilation unit grow because of the interprocedural constant propagation (in percent).",
"На скільки може збільшитися задана компіляційна одиниця через міжпроцедурну поширення констант (у відсотках)."
],
[
"Desktop Cutting Plotter (AutoCAD DXF R12) (*.dxf)",
"Настільний плотер (AutoCAD DXF R12) (*.dxf)"
],
[
"Download cancelled by user",
"Звантаження скасовано користувачем"
],
[
"ManagedObjectReference is missing 'type' property",
"У ManagedObjectReference не вистачає властивості «type»"
],
[
"Use DT_NEEDED only for shared libraries that are used",
"встановлювати DT_NEEDED лише для спільних бібліотек, які використовуються"
],
[
"Can't add stop bit to mark end of instruction group",
"Не вдалося додати біт зупинки для позначення кінця групи інструкцій"
],
[
"Show in blocked plug-in",
"Показати у заблокованому додатку"
],
[
".end [no-]density is ignored",
"директиву .end [no-]density проігноровано"
],
[
"unable to write parameters to config file",
"не вдалося записати параметри до конфігураційного файлу"
],
[
"List requested locales (languages codes).",
"Список запитаних локалей (мовних кодів)."
],
[
"__VA_OPT__ can only appear in the expansion of a C++20 variadic macro",
"__VA_OPT__ може з'являтися лише у розширенні варіативних макросів зі стандарту C++20"
],
[
"Control 2 - <b>Ctrl+Alt+Click</b>: reset, <b>Ctrl</b>: move along axes",
"Керування 2 — <b>Ctrl+Alt+клацання</b>, щоб скинути, <b>Ctrl</b>, щоб рухатися вздовж осей"
],
[
"Failed to convert the command string to argv-lists",
"Не вдалося перетворити рядок команди на списки argv"
],
[
"Usage: @SCDAEMON@ [options] (-h for help)",
"Використання: @SCDAEMON@ [параметри] (-h — довідка)"
],
[
"Mark object to interpose all DSOs but executable",
"позначити об'єкт як такий, що перериває всі DSO, окрім виконуваних"
],
[
"Failed to initialize systemd-journal watch",
"Не вдалося ініціалізувати спостереження systemd-journal"
],
[
"Incorrect magic values in the journal header.",
"Неправильні сигнатури у заголовку журналу."
],
[
"If TRUE, the child will not be subject to homogeneous sizing",
"Якщо TRUE, підпрограма не буде піддаватись однорідному змінюванню розміру"
],
[
"Run a reboot command in the target domain.",
"Виконати команду reboot цільового домену."
],
[
"ivshmem device is no longer supported",
"підтримку пристроїв ivshmem припинено"
],
[
"The installer cannot continue due to a critical error: $0",
"Продовження встановлення неможливе через критичну помилку: $0"
],
[
"rebuild database inverted lists from installed package headers",
"перебудувати зворотні списки бази даних на основі заголовків встановлених пакунків"
],
[
"Unable to open source file for reading",
"Неможливо відкрити файл для читання"
],
[
"Combine sides (reverse)",
"З’єднувати збоку (у зворотному порядку)"
],
[
"Cannot get host interface addresses",
"Не вдалося отримати адреси інтерфейсів основної системи"
],
[
"If TRUE, BSD compression will not be requested.",
"Якщо TRUE, стискання BSD не вимагатиметься."
],
[
"Quechua, Cajatambo North Lima",
"кечуа (Кахатамбо, північна Ліма)"
],
[
"Create a color managed device",
"Створити пристрій з керуванням кольорами"
],
[
"cannot modify virtio network device driver options",
"не вдалося змінити параметри драйвера пристрою мережі virtio"
],
[
"Warn about code paths in which a read or write is performed on a closed file descriptor.",
"Попереджувати про шляхи коду, в яких виконується читання або запис на закритому файловому дескрипторі."
],
[
"Read 8-bit value from ADDR.",
"Прочитати 8-бітове значення за АДРЕСОЮ."
],
[
"preserve group vector instead of setting to target's",
"зберегти вектор групи, не встановлювати вектор вказаного користувача"
],
[
"Bug.search(quicksearch) return value did not contain member 'bugs'",
"Bug.search(quicksearch) повернуто значення, у якому не містилося елемента «bugs»"
],
[
"failed to read AppArmor template",
"не вдалося прочитати дані шаблону AppArmor"
],
[
"Pressed metal with a rolled edge",
"Тиснений метал з прокатаним краєм"
],
[
"Correction according to film type",
"Виправлення відповідно до типу плівки"
],
[
"Generate extended arithmetic instructions, only valid for ARC700.",
"Генерувати розширені арифметичні інструкції, дійсні лише для ARC700."
],
[
"Show version 1 tables only.",
"Показати лише таблиці версії 1."
],
[
"Sets the device vendor",
"Встановити назву виробника пристрою"
],
[
"client which to retrieve identity information for",
"клієнт, для якого слід отримати дані щодо профілю"
],
[
"Reboot the system while an application is inhibiting this",
"Перезапуск системи, коли програми намагаються перешкодити цьому"
],
[
"canceling the wait for synchronous replication and terminating connection due to administrator command",
"скасування очікування синхронної реплікації і завершення з'єднання по команді адміністратора"
],
[
"The type of animation used to transition",
"Тип анімації, яку буде використано для переходу"
],
[
"Don't attach connectors to text objects",
"Не приєднувати лінії з'єднання до текстових об'єктів"
],
[
"Trace bitmap",
"Векторизація растрового зображення"
],
[
"custom monitor control commands issued",
"надіслано типові команди керування монітором"
],
[
"Max number of bytes to compare without loops.",
"Максимальна кількість байтів для порівняння без циклів."
],
[
"Bosnian (with guillemets)",
"Боснійська (з кутовими лапками)"
],
[
"Failed to load nbd module: administratively prohibited",
"Не вдалося завантажити модуль nbd: це заборонено адміністративно"
],
[
"List SPARQL endpoints available in DBus",
"Вивести список кінцевих точок SPARQL, які доступні у DBus"
],
[
"<span size='large'>Create snapshot</span>",
"<span size='large'>Створення знімка</span>"
],
[
"r2 should not be used in indexed addressing mode",
"r2 слід використовувати у режимі індексованого адресування"
],
[
"Central Time (Northern Territory)",
"Центральний час (Північна Територія)"
],
[
"Select <b>object(s)</b> to paste live path effect to.",
"Виберіть <b>об'єкти</b> для застосування інтерактивного ефекту контуру."
],
[
"Child widget to appear next to the button text",
"Спадкоємний віджет, що з'являтиметься на екрані поруч з текстом кнопки"
],
[
"Valid arguments to -mr10k-cache-barrier=:",
"Допустимі аргументи для -mr10k-cache-barrier=:"
],
[
"do not enable STP for this bridge",
"не вмикати STP для цього містка"
],
[
"Minimal length of a search string",
"Мінімальна довжина рядка пошуку"
],
[
"Work on the system-wide installation (default)",
"Працювати над загальносистемними встановленими даними (типово)"
],
[
"Exposure compensation setting",
"Налаштування компенсації експозиції"
],
[
"Multiple <model> elements in controller definition not allowed",
"Не можна використовувати декілька елементів <model> у визначенні контролера"
],
[
"Use alternative raw meta-data cache directory.",
"Використовувати альтернативну директорію для необроблених метаданих."
],
[
"Horizontal panel layout",
"Горизонтальне компонування панелі"
],
[
"actions configured by installer",
"дії, налаштовані засобом для встановлення"
],
[
"Maximum number of escape points tracked by modref per SSA-name.",
"Максимальна кількість точок виходу, відстежуваних модульним посиланням на кожне SSA-імʼя."
],
[
"Unrecognized image file format",
"Нерозпізнаний формат файлу зображення"
],
[
"GUI|Installation Destination|Filter|Other|ID\u0004Show Only _Devices Containing:",
"Показувати лише _пристрої, що містять:"
],
[
"Software Token Authentication",
"Програмне розпізнавання за ключем"
],
[
"                                   STYLE can be ",
"                                   Можливі значення параметра СТИЛЬ: "
],
[
"weak alias definitions not supported in this configuration",
"слабкі визначення псевдонімів не підтримуються в цій конфігурації"
],
[
"The port on the machine defined by “/system/proxy/https/host” that you proxy through.",
"Порт на машині, визначений у «/system/proxy/https/host, що використовується як проксі."
],
[
"void value not ignored as it ought to be",
"значення void не ігнорується так, як повинно бути"
],
[
"Show raw contents of ATA IDENTIFY sector.",
"Показати вміст сектора ATA IDENTIFY без обробки."
],
[
"partial clone failed; attempting full clone",
"не вдалося зробити розріджений клон; спроба зробити повний клон"
],
[
"Command not found, valid commands are:",
"Невідома команда, відомі команди:"
],
[
"Displays help as you browse the books on the left.",
"Показує довідку у той час, коли ви гортаєте книжки ліворуч."
],
[
"Failed to allocate required memory.",
"Не вдалося отримати потрібний обсяг пам’яті."
],
[
"while reading filesystem superblock.",
"під час читання суперблоку файлової системи."
],
[
"Wrong symbol '{}' in alphanumeric representation: Should be [A-Z, 0-9] or {}",
"Помилковий символ «{}» у літерно-цифровому представленні: має бути [A-Z, 0-9] або {}"
],
[
"Root window protocol version:",
"Версія протоколу головного вікна:"
],
[
"syntax error while parsing &&",
"синтаксична помилка під час обробки &&"
],
[
"TLS_*_S9 relocs are not supported yet",
"Підтримки пересувань TLS_*_S9 ще не передбачено"
],
[
"path to git-upload-pack on the remote",
"шлях до git-upload-pack на віддаленому сервері"
],
[
"Time after which the cursor stops blinking, in seconds.",
"Час, після якого курсор перестає блимати, у секундах."
],
[
"Play from the source matching the specified URI",
"Відтворити з відповідного до вказаного URI джерела"
],
[
"Could not send message: ",
"Не вдалося відіслати повідомлення: "
],
[
"failed to open temporary file",
"не вдалося відкрити тимчасовий файл"
],
[
"Error setting partition type",
"Помилка налаштування типу розділу"
],
[
"It is advised to switch to 'NetworkManager' instead for network management.",
"Радимо вам скористатися для керування мережею програмами з пакунка «NetworkManager»."
],
[
"Second operand to .unwabi must be a constant",
"Другий операнд .unwabi має бути сталим"
],
[
"qemu returned malformed time",
"qemu повернуто значення часу із помилковим форматуванням"
],
[
"No information regarding references to homosexuality",
"Немає відомостей щодо згадування гомосексуальності"
],
[
"Don't show colored output.",
"Не розфарбовувати виведені дані."
],
[
"Write a dependency file listing all files read",
"Записати файл списку залежностей усіх прочитаних файлів"
],
[
"Turn on all upcoming D language features.",
"Увімкнути всі майбутні функції мови D."
],
[
"must specify a notes ref to merge",
"необхідно вказати посилання нотаток для злиття"
],
[
"Generate BTF debug information at default level.",
"Генерувати інформацію для налагодження BTF на рівні за замовчуванням."
],
[
"Hide any objects not given in export-id option",
"Приховати усі об'єкти, які не задано у параметрі export-id"
],
[
"Files do not exist or aren’t indexed",
"Файли не існують або їх не було індексовано"
],
[
"not sending a push certificate since the receiving end does not support --signed push",
"сертифікат надсилання не відправлено, оскільки отримуюча сторона не підтримує --signed push"
],
[
"Japanese (Sun Type 7, PC-compatible)",
"Японська (Sun Type 7, сумісна з ПК)"
],
[
"-msdata=[none,data,sysv,eabi]\tSelect method for sdata handling.",
"-msdata=[none,data,sysv,eabi]\tВиберіть метод обробки sdata."
],
[
"Invalid mount spec",
"Неправильна специфікація монтування"
],
[
"Point-to-Point Tunneling Protocol (PPTP)",
"Point-to-Point Tunneling Protocol (PPTP)"
],
[
"ignore zeroed blocks in archive (means EOF)",
"ігнорувати нульові блоки в архіві (звичайно вказують кінець файла)"
],
[
"ISO C++ did not adopt string literal operator templates taking an argument pack of characters",
"ISO C++ не прийняв операторні шаблони літералів рядків, які приймають набір аргументів з символів"
],
[
"Error storing transaction: {}",
"Помилка при збереженні операції: {}"
],
[
"The autoaudiosink element is missing.",
"Не вказано елемента autoaudiosink."
],
[
"Select new partition table type:",
"Виберіть тип нової таблиці розділів:"
],
[
"unknown devices method",
"невідомий спосіб обробки для пристроїв"
],
[
"Affects system performance and power usage.",
"Впливає на швидкодію системи та споживання енергії."
],
[
"Iteration through all top level section not supported",
"Ітерація усіма розділами верхнього рівня не підтримується"
],
[
"Cancelled via GDBusAuthObserver::authorize-authenticated-peer",
"Скасовано через GDBusAuthObserver::authorize-authenticated-peer"
],
[
"no terminator in the core image",
"у основному образі немає мітки завершення"
],
[
"cgroup cpu is required for scheduler tuning",
"для налаштовування планувальника потрібна cgroup процесора"
],
[
"reboot;restart;",
"reboot;restart;перезавантаження;перезапуск;"
],
[
"You cannot move a file over itself.",
"Неможливо перемістити файл сам у себе."
],
[
"libvirt management daemon:",
"Фонова служба керування libvirt:"
],
[
"Name of the GtkFileChooser backend to use by default",
"Назва GtkFileChooser механізму для типового використання"
],
[
"Specify one or more font files to load.",
"Вказати один або декілька файлів для завантаження."
],
[
"Could not initialize options",
"Не вдалося ініціалізувати параметри"
],
[
"The class of the program used by the camera to set exposure when the picture is taken.",
"Клас програми, використаної фотоапаратом для експонування під час знімання."
],
[
"template parameter packs cannot have default arguments",
"пакети параметрів шаблону не можуть мати аргументів за замовчуванням"
],
[
"Invalid Shift/Extract/Deposit Condition.",
"Некоректна умова Shift/Extract/Deposit."
],
[
"Resource Usage / Asynchronous Behavior",
"Використання ресурсу / Асинхронна поведінка"
],
[
"relocated field and relocation type differ in signedness",
"поле пересування та тип пересування відрізняються за можливістю використання знаку"
],
[
"Timeout for check-alive ping",
"Інтервал між послідовними перевірками працездатності"
],
[
"internal error: unknown hardware resource",
"внутрішня помилка: невідомий апаратний ресурс"
],
[
"The certificate does not match the identity of the site.",
"Сертифікат не відповідає ідентичності сайта."
],
[
"treat commandline arguments as source rpm",
"вважати аргументи рядка команди назвами rpm із кодом пакунків"
],
[
"--disallow-module-loading expects boolean argument",
"Для параметра --disallow-module-loading слід вказувати булевий аргумент"
],
[
"Export Use Hints",
"Експортувати з використанням прив'язок"
],
[
"use FILE to map file owner GIDs and names",
"читати мапу трансляції імен груп та значень GID з ФАЙЛА"
],
[
"Units for the corner radius.",
"Одиниці виміру радіуса закруглення."
],
[
"list supported archive formats",
"показати список підтримуваних форматів архівів"
],
[
"Sorry! There are no details for that application.",
"Вибачте! Не маємо подробиць щодо цієї програми."
],
[
"incorrect event handler string, missing dot",
"помилковий рядок обробника події, відсутня точка"
],
[
"DNS Timeout",
"Перевищено час очікування на DNS"
],
[
"Make all warnings fatal",
"Зробити усі попередження фатальними"
],
[
"Set the target CPU type.",
"Встановити тип цільового процесора."
],
[
"Empty stored backup of selection of objects or nodes",
"Спорожнити збережені резервні копії позначення об'єктів або вузлів"
],
[
"The problem is not of a C/C++ type. Can't install debuginfo",
"Проблема не належить до типу C/C++. Встановлення debuginfo неможливе"
],
[
"No VPN configuration options.",
"Немає параметрів налаштування VPN."
],
[
"Set if the value is inherited by default",
"Вказати, якщо значення типово успадковано "
],
[
"Whether to draw compositing borders and repaint counters",
"Визначає, чи слід малювати композитні рамки і перемальовувати лічильники"
],
[
"Enable barrel shift instructions.",
"Увімкнути інструкції револьверного зсуву."
],
[
"Invalid MS property section",
"Некоректний розділ властивості MS"
],
[
"Change blur/blend filter",
"Змінити фільтр розмивання/змішування"
],
[
"OpenVPN is a popular and flexible free-software VPN solution.",
"OpenVPN — популярне і гнучке вільне рішення для VPN."
],
[
"Cannot import because the key is invalid",
"Не вдалося імпортувати, оскільки ключ є некоректним"
],
[
"Gambling using real money",
"Азартні ігри з використанням справжніх грошей"
],
[
"Generate code for the supervisor mode (default).",
"Генерувати код для режиму супервізора (за замовчуванням)."
],
[
"Authentication is required to disconnect a NVMe over Fabrics controller $(drive)",
"Щоб отримати доступ до від'єднання NVMe за допомогою контролера Fabrics $(drive), слід пройти розпізнавання)"
],
[
"List of actions (with optional arguments) to execute",
"Список дій (із додатковими аргументами), які слід виконати"
],
[
"W_ith more often played songs first",
"_В першу чергу найчастіше відтворювані доріжки"
],
[
"Check if FILE is x86_64 kNetBSD",
"Перевірити, чи є ФАЙЛ файлом x86_64 kNetBSD"
],
[
"invalid pass positioning operation",
"недійсна операція позиціонування проходу"
],
[
"unexpected null values in result while fetching remote files",
"неочікувані нульові значення в результаті при отриманні віддалених файлів"
],
[
"make replay advance given branch",
"зробити відтворення з просуванням даної гілки"
],
[
"Whether the surface should match the allocation",
"Визначає, чи має поверхня відповідати виділеній для неї області"
],
[
"Warn when overload promotes from unsigned to signed.",
"Попереджати, коли перевантаження приводить від беззнакового до знакового."
],
[
"invalid Authentication Handle for SecurID",
"некоректний дескриптор розпізнавання для SecurID"
],
[
"Method to calculate the fillet or chamfer",
"Спосіб обчислення окрайка або фаски"
],
[
"Public key signing has failed.",
"Не вдалося підписати відкритим ключем."
],
[
"Specify new repository, can be used multiple times",
"Вказати нове сховище, можна використовувати декілька разів"
],
[
"Warn if testing floating point numbers for equality.",
"Попереджувати, якщо тестування дробових чисел на рівність."
],
[
"Unable to get domain capabilities",
"Не вдалося отримати дані щодо можливостей домену"
],
[
"BMX instructions are only supported with R2 architecture",
"Інструкції BMX підтримуються лише з архітектурою R2"
],
[
"The child pack direction of the menubar",
"Напрям пакування дочірнього меню у панелі меню"
],
[
"Filesystem is missing ext_attr or inline_data feature",
"У файловій системі не передбачено можливості ext_attr або inline_data"
],
[
"Low-level Commands / Interrogators",
"Низькорівневі команди / Допитувачі"
],
[
"Symbolic size to use for named icon",
"Символьний розмір, використовуваний для іменованої піктограми"
],
[
"Method to use to track mouse events",
"Спосіб відстеження подій, пов’язаних із мишею"
],
[
"PID namespace support is required",
"Потрібна підтримка простору назв PID"
],
[
"Ignore differences in build ID",
"Ігнорувати відмінності у ідентифікаторі збирання"
],
[
"further warnings about FDE encoding preventing .eh_frame_hdr generation dropped",
"подальші попередження щодо кодування FDE, яке заважає створенню .eh_frame_hdr, пропущено"
],
[
"retrieve client's identity info from server",
"отримати інформацію щодо профілю клієнта з сервера"
],
[
"addend used with $DSBT_INDEX",
"доданок, використаний з $DSBT_INDEX"
],
[
"expected alignment after size",
"після розміру мало бути вказано вирівнювання"
],
[
"PAM stack to use",
"Стек PAM, який слід використовувати"
],
[
"Could not load IPv4 user interface.",
"Не вдалось завантажити інтерфейс користувача налаштовування IPv4"
],
[
"Enable TPF-OS tracing code.",
"Увімкнути код відстеження TPF-OS."
],
[
"Ignoring surplus option -p",
"Ігнорується надлишкова опція -p"
],
[
"page unselected to cursor position",
"знято виділення сторінки до позиції курсору"
],
[
"Southeastern Ixtlán Zapotec",
"сапотецька (південно-східний Іштлан)"
],
[
"Copied an emoji to your clipboard.",
"Скопійовано емодзі до вашого буфера обміну даними."
],
[
"  a qualified-id is required",
"  потрібно вказати кваліфікований ідентифікатор"
],
[
"'#' is not followed by a macro parameter",
"'#' не супроводжується параметром макроса"
],
[
"Could not spawn pack-objects",
"Не вдалося розмножити об’єкти пакунків"
],
[
"Pedantic checking of ELF files compliance with gABI/psABI spec.",
"Педантична перевірка файлів ELF на сумісність зі специфікаціями gABI/psABI."
],
[
"fr_mem record before region record!",
"Запис fr_mem перед записом області!"
],
[
"Use the button below to create your first snapshot.",
"Натисніть кнопку нижче, щоб створити ваш перший знімок."
],
[
"GIN pending list cannot be cleaned up during recovery.",
"Черга записів GIN не може бути очищена під час відновлення."
],
[
"Could not set role in ibpkey context for {subnet_prefix}/{pkey}",
"Не вдалося встановити роль у контексті ibpkey для {subnet_prefix}/{pkey}"
],
[
"unable to set runas group vector",
"не вдалося встановити вектор групи виконання"
],
[
"unexpected result set after end-of-streaming",
"неочікуваний набір результатів після кінця передачі"
],
[
"Take snapshots of virtual machines to restore to previous states",
"Створення знімків віртуальних машин для відновлення їхніх попередніх станів"
],
[
"Debug logging was already enabled.",
"Журнал налагодження вже увімкнено."
],
[
"Error parsing option --gdk-no-debug",
"Помилка аналізування параметра --gdk-no-debug"
],
[
"Failed to unload shared library",
"Не вдалося вивантажити бібліотеку спільного використання"
],
[
"Cannot make libdeps object readable.",
"Не вдалося зробити об'єкт libdeps придатним до читання."
],
[
"The feed does not contain any downloadable items",
"Потік не містить елементів, придатних для звантаження"
],
[
"Size of the cursor used as cursor theme.",
"Розмір вказівника, що використовується у якості теми вказівника."
],
[
"Volume key is too small for encryption with integrity extensions.",
"Ключ тому є надто малим для шифрування із розширеннями цілісності."
],
[
"-iplugindir=<dir>\tSet <dir> to be the default plugin directory.",
"-iplugindir=<кат>\tВстановити <кат> як каталог за замовчуванням для плагінів."
],
[
"Set network totals unit separately",
"Встановити одиницю даних мережі окремо"
],
[
"Unrecognized or unsupported floating point constant",
"Невідома або непідтримувана константа з рухомою крапкою"
],
[
"comparison of pointers to disjoint address spaces",
"порівняння вказівників на розʼєднані адресні простори"
],
[
"control file contains invalid checkpoint location",
"контрольний файл містить недійсне розташування контрольної точки"
],
[
"Warn if a deprecated compiler feature, class, method, or field is used.",
"Попереджати, якщо використовується застаріла функція компілятора, клас, метод або поле."
],
[
"whole row unique index inference specifications are not supported",
"вказівки з посиланням на весь рядок для вибору унікального індексу не підтримуються"
],
[
"Whether the layout should be vertical, rather than horizontal",
"Визначає, чи має бути компонування вертикальним, замість горизонтального"
],
[
"Load earlier alternatives",
"Завантажити попередні альтернативи"
],
[
"Garbage found at the end of client-final-message.",
"Сміття знайдено в кінці останнього повідомлення клієнта."
],
[
"user to set authorized keys for",
"користувач, для якого слід встановити список уповноважених ключів"
],
[
"Unable to get daemon logging filters information",
"Не вдалося отримати дані щодо фільтрів журналу"
],
[
"Show process “Unit” column on startup",
"Показувати під час запуску стовпчик одиниці процесу"
],
[
"The rotation center on the Z axis",
"Координата центра обертання за віссю Z"
],
[
"Modifier to use for extended window management operations",
"Модифікатор, що використовується для розширених дій віконного менеджера"
],
[
"Unable to parse configuration",
"Не вдалося обробити налаштування"
],
[
"adapter wwnn to be used for underlying storage",
"wwnn адаптера, яку буде використано для основного сховища даних"
],
[
"Fit the image to the window",
"Розташувати зображення цілком у вікні"
],
[
"Turkish, Ottoman (1500-1928)",
"оттоманська турецька (1500-1928)"
],
[
"Can’t handle the supplied version of the icon encoding",
"Не вдалося обробити вказану версію кодування піктограми"
],
[
"Focal Plane Resolution Unit",
"Одиниця роздільності у фокальній площині"
],
[
"Authentication is required to set the system locale.",
"Для визначення системної локалі слід пройти розпізнавання."
],
[
"every bound following MAXVALUE must also be MAXVALUE",
"за кожною границею MAXVALUE повинні бути лише границі MAXVALUE"
],
[
"This could be either a client-software bug or evidence of an attempted man-in-the-middle attack.",
"Це може бути або помилкою клієнтського програмного забезпечення, або доказом спроби техносферної атаки."
],
[
"Can access devices such as webcams or gaming controllers",
"Може отримувати доступ до пристроїв, подібних до вебкамер та ігрових контролерів"
],
[
"Portuguese (Brazil, Dvorak)",
"Португальська (Бразилія, Дворак)"
],
[
"If true, shape Arabic text.",
"Якщо «true», форматувати текст арабською."
],
[
"Generic PCL 6 Tabl Printer wide margin",
"Типовий принтер PCL 6 Tabl із широкими полями"
],
[
"hint in B unit can't be used",
"не можна використовувати підказку у модулі B"
],
[
"(DP) offset out of range.",
"(DP) перевищення можливого зміщення."
],
[
"Enter TPM2 parent key password:",
"Введіть пароль до батьківського ключа TPM2:"
],
[
"Sort direction the sort indicator should indicate",
"Напрямок впорядкування слід показувати індикатором"
],
[
"Equal parallel destination registers, one result will be discarded",
"Однакові паралельні регістри призначення. Один результат буде відкинуто."
],
[
"<aliases> already specified for this key",
"<aliases> для цього ключа вже вказано"
],
[
"Use instructions of and schedule code for given CPU.",
"Використовуйте інструкції та розкладайте код для вказаного процесора."
],
[
"--key-description parameter is mandatory for token add action.",
"Параметр --key-description є обов'язковим для дій із додавання жетонів."
],
[
"the content of uninitialized storage is not usable in a constant expression",
"вміст неініціалізованого сховища не може бути використаний в сталому виразі"
],
[
"shortcut window\u0004Stop scan in progress",
"Припинення поточного сканування"
],
[
"packed driver option is only supported for virtio devices",
"підтримку параметра драйвера packed передбачено лише для пристроїв virtio"
],
[
", drag to adjust, middle-click to remove",
", перетягніть, щоб змінити, клацніть середньою кнопкою, щоб вилучити"
],
[
"Genius Comfy KB-16M/Multimedia KWD-910",
"Genius Comfy KB-16M/Multimedia KWD-910"
],
[
"translator-credits",
"Юрій Чорноіван <yurchor@ukr.net>, 2020"
],
[
"Bad linked list in profile structures",
"Помилковий зв’язаний список у структурах профілю"
],
[
"could not finish pack-objects",
"не вдалося завершити pack-objects"
],
[
"Name of active Tx balancer. Active Tx balancing is disabled by default.",
"Назва балансувальника активного Tx. Типово, балансування активного Tx вимкнено."
],
[
"maxnames > REMOTE_DOMAIN_SNAPSHOT_LIST_MAX",
"maxnames > REMOTE_DOMAIN_SNAPSHOT_LIST_MAX"
],
[
"_Stop Multi-disk Device",
"З_упинити багатодисковий пристрій"
],
[
"Authentication is required to modify a device plugged into another seat",
"Щоб отримати доступ до внесення змін до пристрою, з’єднаного з іншого місця, слід пройти розпізнавання"
],
[
"The field is read-only",
"Поле є придатним лише для читання"
],
[
"Authentication is required to perform a sanitize operation of $(drive)",
"Щоб отримати доступ до дії із очищення пристрою $(drive), слід пройти розпізнавання"
],
[
"Warn if a user-procedure has the same name as an intrinsic.",
"Попереджати, якщо користувацька процедура має ту саму назву, що і вбудована."
],
[
"query information about the guest (via agent)",
"надіслати запит щодо відомостей про гостьову систему (за допомогою агента)"
],
[
"input file does not appear to be a valid archive (too short?)",
"вхідний файл не схожий на архівний (закороткий?)"
],
[
"vhost-user-gpu failed to start",
"Не вдалося запустити vhost-user-gpu"
],
[
"reading of section headers failed",
"спроба читання заголовків розділів зазнала невдачі"
],
[
"Invalid character class name",
"Некоректна назва класу символів"
],
[
"error starting mount daemon",
"помилка запуску служби монтування"
],
[
"Could not set user in port context for {proto}/{port}",
"Не вдається вказати користувача у контексті порту для {proto}/{port}"
],
[
"could not find any WAL files",
"не вдалося знайти ні одного файла WAL"
],
[
"Could not create openwsman client",
"Не вдалося створити клієнт openwsman"
],
[
"unable to update temporary index",
"не вдалося оновити тимчасовий індекс"
],
[
"Unable to allocate instance id",
"Не вдалося розмістити у пам'яті ідентифікатор екземпляра"
],
[
"a nested namespace definition cannot have attributes",
"визначення вкладеного простору імен не може мати атрибутів"
],
[
"Cannot get clipboard data. Clipboard data changed before we could get it.",
"Не вдалося отримати дані буфера обміну. Дані у буфері обміну було змінено до того, як ми змогли їх отримати."
],
[
"Error reading mimetype xml file",
"Помилка під час читання файла xml типу MIME"
],
[
"Could not find 'active' element",
"Не вдалося знайти елемент «active»"
],
[
"Unable to format NUMA node cache",
"Не вдалося виконати форматування кешу вузлів NUMA"
],
[
"cannot restore from compressed archive (compression not supported in this installation)",
"не вдалося відновити зі стиснутого архіву (встановлена версія не підтримує стискання)"
],
[
"Error: Unable to Show User Settings",
"Помилка: не вдалося показати параметри користувача"
],
[
"failed to get cgroup BPF prog FD",
"не вдалося отримати файловий дескриптор програми BPF cgroup"
],
[
"invalid merge entity size",
"некоректний розмір запису об’єднання"
],
[
"Failed to generate microreport from the problem data",
"Не вдалося створити мікрозвіт на основі даних щодо проблеми"
],
[
"Erasing the data cannot be undone. Be sure to have backups.",
"Наслідки дії з витирання даних є незворотними. Не забудьте створити резервні копії."
],
[
"Error writing to log",
"Помилка під час запису до журналу"
],
[
"Error creating print preview",
"Помилка при створенні попереднього перегляду"
],
[
"cannot enable executable stack as shared object requires",
"не вдалося увімкнути стек виконання, як цього вимагає об’єкт спільного використання"
],
[
"cannot modify network device tap name",
"неможливо змінити tap-назву пристрою мережі"
],
[
"cannot list vcpu pinning for an inactive domain",
"не можна побудувати список прив'язування віртуальних процесорів для неактивного домену"
],
[
"SOAP: Failed to add new issue parameters because the required items are missing.",
"SOAP: не вдалося додати нові параметри вади, оскільки немає потрібних для цього записів."
],
[
"unable to fdopen alternates lockfile",
"не вдалося виконати fdopen для файла блокування запозичених обʼєктів"
],
[
"failed to create socket for remote",
"не вдалося створити сокет для remote"
],
[
"defining a type in a cast is invalid in C++",
"визначення типу в приведенні недійсне в C++"
],
[
"Unable to retrieve threadpool parameters",
"Не вдалося отримати параметри threadpool"
],
[
"invalid driver type for version detection",
"Некоректний тип для виявлення версії"
],
[
"Could not retrieve resource pool",
"Не вдалося отримати буфер ресурсів"
],
[
"permission denied to finish prepared transaction",
"немає дозволу для завершення підготовлених транзакцій"
],
[
"cannot enable ipv4.link-local with ipv4.method=disabled",
"не можна увімкнути ipv4.link-local з ipv4.method=disabled"
],
[
"Canon EOS Capture failed to release: Perhaps no focus?",
"Не вдалося увімкнути знімання Canon EOS: можливо, немає фокусування?"
],
[
"cannot allocate buffer for object name",
"не вдалося отримати пам’ять для назви об’єкта"
],
[
"Unable to allocate a dasd disklabel slot",
"Не вдається розподілити слот етикетки диска dasd"
],
[
"The transaction failed",
"Спроба виконання операції зазнала невдачі"
],
[
"unable to load compose definitions because some of them are too large",
"не вдалося завантажити визначення композицій, оскільки деякі з них є надто великими"
],
[
"could not find variant declaration",
"не вдалося знайти оголошення варіанту"
],
[
"cannot use more than one of -anrw",
"-anrw можуть зустрічатися лише один раз"
],
[
"Failed to decrypt the private key: unexpected padding length.",
"Не вдалося розшифрувати закритий ключ: неочікувана довжина доповнення."
],
[
"Could not set memory size",
"Не вдалося встановити об'єм пам'яті"
],
[
"Error handling synchronous load with custom protocol",
"Помилка під час обробки синхронного навантаження із нетиповим протоколом"
],
[
"Failed to process keyinfo file: ",
"Не вдалося обробити файл keyinfo: "
],
[
"Invalid syntax in External addressing mode",
"Некоректна синтаксична конструкція у режимі зовнішнього адресування"
],
[
"Unable to get hypervisor name",
"Не вдалося отримати назву гіпервізора"
],
[
"Unable to disable nagle algorithm",
"Не вдалося вимкнути використання алгоритму nagle"
],
[
"<vendor_field> evaluation has failed",
"Помилка під час обчислення <vendor_field>"
],
[
"cannot specify HEADER in BINARY mode",
"не можна вказати HEADER у режимі BINARY"
]
]

corpus = load_system_corpus()
USING_FALLBACK = len(corpus) < 5000
if USING_FALLBACK:
    corpus = [('fallback', s, t) for s, t in FALLBACK_PAIRS]
    print("⚠️  УВАГА: української локалі на цій машині немає або вона майже порожня.")
    print("⚠️  Увімкнено вбудований запасний корпус на", len(corpus), "документів.")
    print("⚠️  Усі числа нижче будуть ІНШІ, ніж у лекції. Це не помилка зошита.")
else:
    print("корпус зібрано з /usr/share/locale/uk/LC_MESSAGES/*.mo")

programs = sorted({p for p, s, t in corpus})
print("документів:", len(corpus))
print("програм   :", len(programs))
print()
print("як виглядає один запис:")
program, source, target = corpus[17]
print("  програма :", program)
print("  оригінал :", source)
print("  переклад :", target)

## 3 · Мітка з англійського боку

Задача теми: **чи повідомляє цей рядок про помилку**. Мітку ми беремо не руками, а
регуляркою по **англійському оригіналу** — вісім типових слів, якими англомовні
програми повідомляють про збій.

Ознаки при цьому будуть із **українського перекладу**. Модель не бачить того боку,
з якого зроблено мітку.

In [ ]:
ERROR_WORDS = re.compile(
    r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b', re.I)

sources = [source for program, source, target in corpus]      # англійський бік: мітка
documents = [target for program, source, target in corpus]    # український бік: ознаки
labels = np.array([1 if ERROR_WORDS.search(s) else 0 for s in sources])

print("документів      :", len(documents))
print("клас «помилка»  :", int(labels.sum()), f"= {labels.mean():.4f}")
print("клас «не помилка»:", int((labels == 0).sum()), f"= {(labels == 0).mean():.4f}")
print()
print("по два приклади кожного класу:")
shown = {0: 0, 1: 0}
for i in range(len(corpus)):
    y_i = labels[i]
    if shown[y_i] >= 2:
        continue
    shown[y_i] += 1
    found = ERROR_WORDS.search(sources[i])
    print(f"  мітка {y_i} ({'слово ' + repr(found.group(0)) if found else 'жодного слова зі списку'})")
    print("     англ:", sources[i][:78])
    print("     укр :", documents[i][:78])

## 4 · Чи чесна така мітка

Головне заперечення до цієї задачі звучить так: «ви зробили мітку з тексту й на
цьому ж тексті вчитесь — це замкнене коло». Заперечення слушне, і його треба
закрити числом, а не словом.

Перевіряємо дві речі. По-перше, чи не просочилося англійське ключове слово в
український переклад — тоді модель могла б прочитати мітку просто так. По-друге,
чи не звʼязана мітка з простою поверхневою ознакою на кшталт «у рядку є латиниця».

In [ ]:
latin_key = ERROR_WORDS                       # той самий шаблон, але тепер по українському боку
leaked = [d for d in documents if latin_key.search(d)]
print(f"українських документів із англійським ключовим словом: {len(leaked)}"
      f"  ({len(leaked) / len(documents) * 100:.4f} %)")

# чи не є мітка просто «в рядку є латиниця»
has_latin = np.array([bool(re.search(r'[A-Za-z]{3,}', d)) for d in documents])
print(f"українських документів із будь-яким латинським словом: {int(has_latin.sum())}"
      f"  ({has_latin.mean() * 100:.2f} %)")
print(f"  частка «помилка» серед них        : {labels[has_latin].mean():.4f}")
print(f"  частка «помилка» серед решти      : {labels[~has_latin].mean():.4f}")
print()
print("Обидва числа майже однакові — отже «є латиниця» мітки не пояснює.")

## 5 · Дублікати: те, що ламає поділ вибірки мовчки

Різні програми беруть однакові рядки з однакових бібліотек — тож той самий текст
трапляється в корпусі по кілька разів. Якщо копії одного тексту потраплять і в
навчання, і в перевірку, оцінка буде завищена: модель відповідає на питання, яке
вже бачила з відповіддю.

Спершу порахуємо, скільки таких копій узагалі є.

In [ ]:
text_counts = collections.Counter(documents)
unique_texts = len(text_counts)
extra_copies = len(documents) - unique_texts
docs_in_dupe_groups = sum(c for t, c in text_counts.items() if c > 1)

print(f"документів                        : {len(documents)}")
print(f"унікальних текстів                : {unique_texts}")
print(f"зайвих копій                      : {extra_copies}"
      f"  ({extra_copies / len(documents) * 100:.2f} %)")
print(f"документів у групах-повторах      : {docs_in_dupe_groups}"
      f"  ({docs_in_dupe_groups / len(documents) * 100:.2f} %)")

# скільки з них ділять кілька РІЗНИХ програм — це і є перетин між програмами
programs_of_text = collections.defaultdict(set)
for program, source, target in corpus:
    programs_of_text[target].add(program)
cross_docs = sum(c for t, c in text_counts.items() if c > 1 and len(programs_of_text[t]) > 1)
print(f"з них розділені між різними програмами: {cross_docs}"
      f"  ({cross_docs / len(documents) * 100:.2f} %)")
print()
print("три найчастіші повтори:")
for text, count in text_counts.most_common(3):
    print(f"  ×{count}  {text[:64]}")

## 6 · Слова, ваги й поділ

Токенізатор — спільний канон блоку. Ваги — TF-IDF із теми 05. Поділ — 70 % на
навчання, 30 % на перевірку, і **три зерна**: 0, 1 і 2.

`min_df=2` викидає слова, що трапились в одному-єдиному документі. Це не оптимізація
пам'яті: слово, побачене раз, не дає моделі нічого, крім можливості запамʼятати
цей один документ.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"     # канон блоку, тема 04

n_docs = len(documents)
splits = {}
for seed in (0, 1, 2):
    train_idx, test_idx = train_test_split(np.arange(n_docs), test_size=0.3, random_state=seed)
    splits[seed] = (train_idx, test_idx)

train_idx, test_idx = splits[0]
vectorizer = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
X_train = vectorizer.fit_transform([documents[i] for i in train_idx])
X_test = vectorizer.transform([documents[i] for i in test_idx])

print("навчальних документів :", X_train.shape[0])
print("перевірних документів :", X_test.shape[0])
print("ознак (слів)          :", X_train.shape[1])
print("ненульових клітинок   :", X_train.nnz)
print(f"заповнено             : {X_train.nnz / (X_train.shape[0] * X_train.shape[1]) * 100:.4f} %")
print(f"слів на документ (сер.): {X_train.nnz / X_train.shape[0]:.2f}")

## 7 · Як тут міряти час

Далі ми будемо порівнювати моделі за швидкістю, і тут є пастка. `time.perf_counter()`
міряє час на стінному годиннику — тобто разом із тим, скільки процесор витратив на
**чужі** програми. На завантаженій машині одне й те саме навчання «триває» то дві
секунди, то пʼять, і жодне з цих чисел не про модель.

`time.process_time()` міряє **процесорний час самого процесу**. Він не залежить від
того, що ще крутиться поруч, і повторюється від прогону до прогону.

Заміряємо обидва на одному навчанні — і порівняємо.

In [ ]:
def fit_and_time(prototype, X, y, repeats=3):
    """Навчаємо кілька разів і беремо НАЙМЕНШИЙ процесорний час.
    Мінімум — найчесніша оцінка: завадити заміру чуже навантаження може
    тільки в один бік, а саме збільшити його.
    `clone` робить свіжу ненавчену копію моделі — щоб кожен повтор
    починався з нуля, а не донавчав попередню."""
    best_cpu, best_model = float('inf'), None
    for _ in range(repeats):
        model = clone(prototype)
        start = time.process_time()
        model.fit(X, y)
        spent = time.process_time() - start
        if spent < best_cpu:
            best_cpu, best_model = spent, model
    return best_model, best_cpu


y_train = labels[train_idx]
for attempt in range(3):
    model = LogisticRegression(max_iter=1000)
    wall_start, cpu_start = time.perf_counter(), time.process_time()
    model.fit(X_train, y_train)
    wall = time.perf_counter() - wall_start
    cpu = time.process_time() - cpu_start
    print(f"прогін {attempt + 1}: стінний час {wall:6.3f} с | процесорний {cpu:6.3f} с"
          f" | відношення ×{wall / cpu:5.2f}")
print()
print("Стінний час стрибає, процесорний — ні. Далі скрізь друкуємо процесорний.")

## 8 · Головна таблиця теми

Чотири моделі, три зерна. Дивимось на **F1 по класу «помилка»** — тобто по
меншому класу, який нас і цікавить. Точність (accuracy) тут дивитись не можна, і
чому саме — розберемо в самому кінці зошита.

`n_iter_` у логістичної регресії — це скільки кроків зробив розвʼязувач `lbfgs`,
поки не зійшовся. Число нам ще знадобиться.

In [ ]:
MODELS = [
    ("MultinomialNB",   MultinomialNB()),
    ("ComplementNB",    ComplementNB()),
    ("LogReg",          LogisticRegression(max_iter=1000)),
    ("LogReg balanced", LogisticRegression(max_iter=1000, class_weight='balanced')),
]

results = collections.defaultdict(list)
matrices = {}          # знадобляться далі, щоб не векторизувати вдруге
predictions = {}

for seed in (0, 1, 2):
    tr, te = splits[seed]
    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    Xtr = vec.fit_transform([documents[i] for i in tr])
    Xte = vec.transform([documents[i] for i in te])
    matrices[seed] = (vec, Xtr, Xte)
    for name, prototype in MODELS:
        model, cpu = fit_and_time(prototype, Xtr, labels[tr])
        pred = model.predict(Xte)
        score = f1_score(labels[te], pred)
        steps = int(model.n_iter_[0]) if hasattr(model, 'n_iter_') else None
        results[name].append((score, cpu, steps))
        predictions[(seed, name)] = pred

print(f"{'модель':<17}{'F1 «помилка»':>16}{'проц. час, с':>15}{'ітерацій':>12}")
for name, _ in MODELS:
    scores = np.array([s for s, c, k in results[name]])
    times = np.array([c for s, c, k in results[name]])
    steps = [k for s, c, k in results[name]]
    step_text = str(steps) if steps[0] is not None else "—"
    print(f"{name:<17}{scores.mean():>9.4f} ±{scores.std():.4f}{times.mean():>15.3f}{step_text:>12}")

Три числа з цієї таблиці варто назвати вголос — розкид по зернах підказує, які
різниці справжні, а які ні.

In [ ]:
mnb = np.array([s for s, c, k in results["MultinomialNB"]])
cnb = np.array([s for s, c, k in results["ComplementNB"]])
lr = np.array([s for s, c, k in results["LogReg"]])
lrb = np.array([s for s, c, k in results["LogReg balanced"]])

t_cnb = np.mean([c for s, c, k in results["ComplementNB"]])
t_lr = np.mean([c for s, c, k in results["LogReg"]])
t_lrb = np.mean([c for s, c, k in results["LogReg balanced"]])

def verdict(gap, spread):
    return "різниця доведена" if abs(gap) > 2 * spread else "у межах розкиду — не різниця"

print(f"ComplementNB − MultinomialNB : {cnb.mean() - mnb.mean():+.4f}"
      f"  (найбільший розкид {max(cnb.std(), mnb.std()):.4f}) → {verdict(cnb.mean() - mnb.mean(), max(cnb.std(), mnb.std()))}")
print(f"LogReg − ComplementNB        : {lr.mean() - cnb.mean():+.4f}"
      f"  (найбільший розкид {max(lr.std(), cnb.std()):.4f}) → {verdict(lr.mean() - cnb.mean(), max(lr.std(), cnb.std()))}")
print(f"balanced − LogReg            : {lrb.mean() - lr.mean():+.4f}"
      f"  (найбільший розкид {max(lrb.std(), lr.std()):.4f}) → {verdict(lrb.mean() - lr.mean(), max(lrb.std(), lr.std()))}")
print()
print(f"ComplementNB швидший за LogReg          у {t_lr / t_cnb:.1f} раза")
print(f"balanced швидший за звичайну логістичну у {t_lr / t_lrb:.2f} раза")

## 9 · Скільки насправді коштує витік через дублікати

Тепер повернімось до дублікатів. Модель уже навчена — подивимось окремо на дві
частини перевірної вибірки: на документи, точна копія яких була в навчанні, і на
решту.

In [ ]:
leak_rows = []
seen_masks = {}
for seed in (0, 1, 2):
    tr, te = splits[seed]
    train_texts = set(documents[i] for i in tr)
    seen = np.array([documents[i] in train_texts for i in te])
    seen_masks[seed] = seen
    pred = predictions[(seed, "LogReg balanced")]
    y_te = labels[te]
    leak_rows.append((seen.mean(),
                      f1_score(y_te[seen], pred[seen]),
                      f1_score(y_te[~seen], pred[~seen]),
                      f1_score(y_te, pred)))
    print(f"зерно {seed}: бачених {seen.sum():5d} ({seen.mean() * 100:.2f} %)"
          f" | F1 на бачених {leak_rows[-1][1]:.4f}"
          f" | F1 на нових {leak_rows[-1][2]:.4f}"
          f" | F1 на всіх {leak_rows[-1][3]:.4f}")

leak = np.array(leak_rows)
print()
print(f"частка бачених у перевірній вибірці: {leak[:, 0].mean():.4f}")
print(f"F1 на бачених : {leak[:, 1].mean():.4f}")
print(f"F1 на нових   : {leak[:, 2].mean():.4f}")
print(f"розрив        : {leak[:, 1].mean() - leak[:, 2].mean():+.4f}")

Розрив є, і він великий. Але питання не в ньому, а в тому, наскільки він **зсуває
підсумкове число**. Це залежить від того, яку частку перевірної вибірки займають
бачені документи.

Перевіряємо прямо: збираємо штучні перевірні вибірки однакового розміру, але з
різною часткою «бачених», і рахуємо F1 на кожній. Жодного нового навчання тут не
треба — модель та сама, міняється лише те, на чому її питають.

In [ ]:
LEAK_SHARES = [0.00, 0.06, 0.15, 0.25, 0.40, 0.60, 0.80, 1.00]
SAMPLE = 5000
leak_curve = collections.defaultdict(list)

for seed in (0, 1, 2):
    tr, te = splits[seed]
    seen = seen_masks[seed]
    pred = predictions[(seed, "LogReg balanced")]
    y_te = labels[te]
    idx_seen = np.where(seen)[0]
    idx_new = np.where(~seen)[0]
    rng = np.random.default_rng(seed)
    for share in LEAK_SHARES:
        n_seen = int(round(SAMPLE * share))
        parts = []
        if n_seen:
            parts.append(rng.choice(idx_seen, n_seen, replace=True))
        if SAMPLE - n_seen:
            parts.append(rng.choice(idx_new, SAMPLE - n_seen, replace=True))
        pick = np.concatenate(parts)
        leak_curve[share].append(f1_score(y_te[pick], pred[pick]))

print(f"{'частка дублікатів':>18}{'F1':>10}{'розкид':>10}")
for share in LEAK_SHARES:
    values = np.array(leak_curve[share])
    mark = "  ← стільки в нашому корпусі" if abs(share - 0.06) < 0.001 else ""
    print(f"{share:>18.2f}{values.mean():>10.4f}{values.std():>10.4f}{mark}")

І остаточна перевірка: зробимо **чесний поділ**, у якому всі копії одного тексту
завжди йдуть в один бік. Якщо витік справді дорого коштує, чесний поділ покаже
помітно нижче число.

In [ ]:
unique_sorted = sorted(text_counts)                       # усі різні тексти корпусу
text_id = {text: i for i, text in enumerate(unique_sorted)}
groups = np.array([text_id[t] for t in documents])        # номер тексту для кожного документа

honest_scores, naive_scores = [], []
for seed in (0, 1, 2):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(text_id))
    cut = int(len(text_id) * 0.7)
    in_train = np.zeros(len(text_id), dtype=bool)
    in_train[order[:cut]] = True
    tr = np.where(in_train[groups])[0]
    te = np.where(~in_train[groups])[0]

    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    Xtr = vec.fit_transform([documents[i] for i in tr])
    Xte = vec.transform([documents[i] for i in te])
    model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr, labels[tr])
    honest_scores.append(f1_score(labels[te], model.predict(Xte)))
    naive_scores.append(leak_rows[seed][3])
    print(f"зерно {seed}: наївний поділ {naive_scores[-1]:.4f} | чесний поділ {honest_scores[-1]:.4f}")

honest = np.array(honest_scores); naive = np.array(naive_scores)
print()
print(f"наївний поділ: {naive.mean():.4f} ±{naive.std():.4f}")
print(f"чесний поділ : {honest.mean():.4f} ±{honest.std():.4f}")
print(f"завищення    : {naive.mean() - honest.mean():+.4f}"
      f"  — при розкиді {max(naive.std(), honest.std()):.4f}")

## 10 · Наскільки наївний наївний Баєс

Наївний Баєс припускає, що слова в документі трапляються **незалежно** одне від
одного. Тобто ймовірність побачити два слова разом дорівнює добутку їхніх
окремих імовірностей:

    P(a і b) = P(a) × P(b)

Це припущення можна не обговорювати, а **заміряти**. Беремо сорок найчастіших
слів, для кожної пари рахуємо, у якій частці документів вони стоять разом, і
ділимо на добуток окремих часток. Одиниця — незалежність виконується. Тридцять
одиниць — слова тримаються разом у тридцять разів частіше, ніж мали б.

In [ ]:
binary_vec = CountVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN,
                             binary=True, min_df=2)
B = binary_vec.fit_transform(documents)          # 1, якщо слово є в документі
words = np.array(binary_vec.get_feature_names_out())
doc_freq = np.asarray(B.sum(axis=0)).ravel()
top = np.argsort(-doc_freq)[:40]

Btop = B[:, top].astype(np.float64)
together = (Btop.T @ Btop).toarray()             # у скількох документах пара разом
p_word = doc_freq[top] / len(documents)

pairs = []
for i in range(len(top)):
    for j in range(i + 1, len(top)):
        p_both = together[i, j] / len(documents)
        p_independent = p_word[i] * p_word[j]
        pairs.append((words[top[i]], words[top[j]], p_both, p_independent,
                      p_both / p_independent))

lifts = np.array([r[4] for r in pairs])
print(f"пар слів              : {len(pairs)}")
print(f"медіана відношення    : {np.median(lifts):.3f}   (незалежність дала б 1.000)")
print(f"мінімум / максимум    : {lifts.min():.3f} / {lifts.max():.3f}")
print(f"пар, де відхилення більше ніж у 1.25 раза: {(np.abs(np.log(lifts)) > np.log(1.25)).mean() * 100:.1f} %")
print(f"пар, де відхилення більше ніж удвічі     : {(np.abs(np.log(lifts)) > np.log(2.0)).mean() * 100:.1f} %")
print()
ordered = sorted(pairs, key=lambda r: -r[4])
print("сильніше за все ТЯГНУТЬСЯ одне до одного:")
for a, b, p_both, p_ind, lift in ordered[:5]:
    print(f"  «{a}» + «{b}»: разом у {p_both * 100:.3f} % документів,"
          f" незалежність обіцяла {p_ind * 100:.3f} % → ×{lift:.2f}")
print("сильніше за все ВІДШТОВХУЮТЬСЯ:")
for a, b, p_both, p_ind, lift in ordered[-5:]:
    print(f"  «{a}» + «{b}»: разом у {p_both * 100:.3f} % документів,"
          f" незалежність обіцяла {p_ind * 100:.3f} % → ×{lift:.2f}")

## 11 · `ComplementNB` — це `MultinomialNB` без апріорної ймовірності

`ComplementNB` подають як «наївний Баєс для незбалансованих даних». На двох
класах він виявляється рівно тим самим наївним Баєсом, у якого прибрали один
доданок — логарифм апріорної ймовірності класу.

Перевіримо це `assert`-ом, а не на слово. Твердження таке: ваги слів у
`ComplementNB` — це ваги `MultinomialNB`, помінянi місцями між класами й узяті зі
знаком мінус. А тоді **різниця** ваг між класами в обох моделей однакова
до останнього знака.

In [ ]:
vec0, Xtr0, Xte0 = matrices[0]
tr0, te0 = splits[0]
y_tr0, y_te0 = labels[tr0], labels[te0]

multinomial = MultinomialNB().fit(Xtr0, y_tr0)
complement = ComplementNB().fit(Xtr0, y_tr0)
no_prior = MultinomialNB(fit_prior=False).fit(Xtr0, y_tr0)   # той самий, але без апріорної

assert np.allclose(multinomial.feature_log_prob_[1], -complement.feature_log_prob_[0]), \
    "ваги класу 1 у MultinomialNB мали б дорівнювати мінус вагам класу 0 у ComplementNB"
assert np.allclose(multinomial.feature_log_prob_[0], -complement.feature_log_prob_[1]), \
    "і навпаки"

diff_multinomial = multinomial.feature_log_prob_[1] - multinomial.feature_log_prob_[0]
diff_complement = complement.feature_log_prob_[1] - complement.feature_log_prob_[0]
gap = float(np.abs(diff_multinomial - diff_complement).max())
print(f"найбільша розбіжність різниць ваг: {gap}")
print(f"тобто {gap * 1e15:.1f} · 10^-15 — нуль у межах точності float64")
assert np.allclose(diff_multinomial, diff_complement), "різниці ваг мали б збігтися"
print("✅ ваги слів у двох моделей ті самі")

same = (complement.predict(Xte0) == no_prior.predict(Xte0)).mean()
print(f"передбачення ComplementNB і MultinomialNB(fit_prior=False) збігаються: {same * 100:.2f} %")
assert same == 1.0, "мали б збігтися всі до одного"
print("✅ на двох класах це буквально одна модель")
print()
prior_shift = float(multinomial.class_log_prior_[1] - multinomial.class_log_prior_[0])
print(f"апріорна частка «помилки» в навчанні: {y_tr0.mean():.4f}"
      f"   решта: {1 - y_tr0.mean():.4f}")
print(f"весь доданок, яким вони різняться: {prior_shift:.4f}")
print(f"він же — логарифм шансів апріорної частки: "
      f"{math.log(y_tr0.mean() / (1 - y_tr0.mean())):.4f}")
print()
print(f"F1: MultinomialNB {f1_score(y_te0, multinomial.predict(Xte0)):.4f}"
      f" | ComplementNB {f1_score(y_te0, complement.predict(Xte0)):.4f}"
      f" | MultinomialNB(fit_prior=False) {f1_score(y_te0, no_prior.predict(Xte0)):.4f}")

## 12 · Отже, це не дві моделі, а один важіль

Якщо різниця між ними — один доданок у сумі, то обидві моделі лежать на одній
прямій, і ми можемо проїхатись по ній повзунком. Зсув 0 — це `ComplementNB`.
Зсув, що дорівнює логарифму шансів апріорної частки, — це `MultinomialNB`.
А де оптимум?

Щоб не обманювати себе, оптимум шукаємо **не на перевірній вибірці**: відрізаємо
від навчальної частини пʼяту частину під добір і дивимось на неї.

In [ ]:
SHIFTS = np.round(np.arange(-3.0, 2.01, 0.25), 2)
shift_curve = collections.defaultdict(list)

# крива будується на моделі, навченій на ВСІЙ навчальній частині —
# тій самій, що дала головну таблицю
for seed in (0, 1, 2):
    vec, Xtr, Xte = matrices[seed]
    tr, te = splits[seed]
    nb = ComplementNB().fit(Xtr, labels[tr])
    weight_diff = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
    score_test = Xte @ weight_diff          # логіт «помилка проти не помилка»
    for s in SHIFTS:
        shift_curve[float(s)].append(
            f1_score(labels[te], (score_test + s > 0).astype(int)))

print(f"{'зсув':>7}{'F1 на перевірній':>20}{'розкид':>10}")
for s in SHIFTS:
    values = np.array(shift_curve[float(s)])
    tags = []
    if abs(s) < 0.01:
        tags.append("← ComplementNB")
    if abs(s + 1.25) < 0.01:
        tags.append("← найближче до MultinomialNB")
    print(f"{s:>7.2f}{values.mean():>20.4f}{values.std():>10.4f}   {' '.join(tags)}")

best_on_curve = max(SHIFTS, key=lambda s: np.mean(shift_curve[float(s)]))
print()
print(f"найкращий зсув на цій кривій: {best_on_curve:+.2f}"
      f" → F1 {np.mean(shift_curve[float(best_on_curve)]):.4f}")
print(f"апріорна ймовірність просить  : {prior_shift:+.4f} (це MultinomialNB)")

Оптимум на цій кривій ми підглянули на перевірній вибірці — а так робити не можна.
Дібравши будь-що на перевірній вибірці, ми перестаємо мати перевірну вибірку.
Зробимо чесно: відріжемо від навчальної частини пʼяту частину, доберемо зсув
**на ній** і назвемо число на перевірній, якої добір не бачив.

In [ ]:
chosen_shifts, tuned_scores, plain_scores = [], [], []
for seed in (0, 1, 2):
    tr, te = splits[seed]
    fit_idx, dev_idx = train_test_split(tr, test_size=0.2, random_state=seed)
    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    Xfit = vec.fit_transform([documents[i] for i in fit_idx])
    Xdev = vec.transform([documents[i] for i in dev_idx])
    Xte = vec.transform([documents[i] for i in te])

    nb = ComplementNB().fit(Xfit, labels[fit_idx])
    weight_diff = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]
    score_dev = Xdev @ weight_diff
    score_test = Xte @ weight_diff

    on_dev = [f1_score(labels[dev_idx], (score_dev + s > 0).astype(int)) for s in SHIFTS]
    best_shift = float(SHIFTS[int(np.argmax(on_dev))])
    chosen_shifts.append(best_shift)
    tuned_scores.append(f1_score(labels[te], (score_test + best_shift > 0).astype(int)))
    plain_scores.append(f1_score(labels[te], (score_test > 0).astype(int)))
    print(f"зерно {seed}: дібраний на відкладеній частині зсув {best_shift:+.2f}"
          f" → на перевірній F1 {tuned_scores[-1]:.4f}"
          f" (без зсуву {plain_scores[-1]:.4f})")

print()
print(f"ComplementNB як є        : {np.mean(plain_scores):.4f} ±{np.std(plain_scores):.4f}")
print(f"з чесно дібраним зсувом  : {np.mean(tuned_scores):.4f} ±{np.std(tuned_scores):.4f}")
print(f"виграш від одного числа  : {np.mean(tuned_scores) - np.mean(plain_scores):+.4f}")

## 13 · Коли який Баєс кращий

Якщо вся різниця — в апріорній ймовірності, то й перевага має залежати від того,
наскільки перекошені класи. Перевіряємо: тримаємо розмір навчальної частини
сталим і міняємо лише частку класу «помилка» в ній. Перевірна вибірка при цьому
не міняється — інакше ми міряли б не моделі, а різні задачі.

In [ ]:
BALANCE_SHARES = [0.50, 0.40, 0.30, 0.2123, 0.15, 0.10, 0.05]
TRAIN_SIZE = min(20000, int(len(documents) * 0.15))
balance_rows = collections.defaultdict(lambda: collections.defaultdict(list))

for seed in (0, 1, 2):
    tr, te = splits[seed]
    rng = np.random.default_rng(seed)
    positives = tr[labels[tr] == 1]
    negatives = tr[labels[tr] == 0]
    Xte_cache = None
    for share in BALANCE_SHARES:
        n_pos = int(TRAIN_SIZE * share)
        n_neg = TRAIN_SIZE - n_pos
        if n_pos > len(positives) or n_neg > len(negatives):
            continue
        sub = np.concatenate([rng.choice(positives, n_pos, replace=False),
                              rng.choice(negatives, n_neg, replace=False)])
        vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
        Xsub = vec.fit_transform([documents[i] for i in sub])
        Xte = vec.transform([documents[i] for i in te])
        for name, prototype in (("MultinomialNB", MultinomialNB()),
                                ("ComplementNB", ComplementNB())):
            model = clone(prototype).fit(Xsub, labels[sub])
            balance_rows[share][name].append(f1_score(labels[te], model.predict(Xte)))

print(f"{'частка «помилка»':>18}{'MultinomialNB':>16}{'ComplementNB':>15}{'розрив':>10}")
for share in BALANCE_SHARES:
    if not balance_rows[share]:
        continue
    a = np.array(balance_rows[share]["MultinomialNB"])
    b = np.array(balance_rows[share]["ComplementNB"])
    print(f"{share:>18.4f}{a.mean():>16.4f}{b.mean():>15.4f}{b.mean() - a.mean():>+10.4f}")
print()
print("При 50/50 різниці немає взагалі — апріорна ймовірність там нульова за логарифмом.")

## 14 · Чому `balanced` не лише кращий, а й швидший

У головній таблиці `class_weight='balanced'` виявився і точнішим, і швидшим за
звичайну логістичну регресію. Друге неочевидне: зважування додає роботи на
кожному кроці, а не віднімає.

Розгадка має бути в кількості кроків. Перевіряємо: ділимо процесорний час на
кількість ітерацій `lbfgs`.

In [ ]:
print(f"{'модель':<17}{'ітерацій':>26}{'у середньому':>14}{'проц. час, с':>14}{'час на ітерацію':>18}")
for name in ("LogReg", "LogReg balanced"):
    steps = np.array([k for s, c, k in results[name]], dtype=float)
    times = np.array([c for s, c, k in results[name]])
    per_step = times / steps
    print(f"{name:<17}{str([int(k) for k in steps]):>26}{steps.mean():>14.1f}"
          f"{times.mean():>14.3f}{per_step.mean():>18.4f}")
print()
lr_steps = np.mean([k for s, c, k in results["LogReg"]])
lrb_steps = np.mean([k for s, c, k in results["LogReg balanced"]])
print(f"кроків менше у {lr_steps / lrb_steps:.2f} раза, "
      f"часу менше у {t_lr / t_lrb:.2f} раза — це те саме число")
print("Отже, кожен крок НЕ став дешевшим. Кроків стало менше.")

Тепер розгортка: не два значення ваги, а шість. Якщо причина справді у ваговому
коефіцієнті, кількість ітерацій має падати разом із її зростанням.

In [ ]:
WEIGHTS = [1.0, 1.5, 2.0, 2.36, 3.0, 3.7115]   # 3.7115 — це і є 'balanced' на наших частках
weight_rows = collections.defaultdict(lambda: collections.defaultdict(list))

for seed in (0, 1, 2):
    vec, Xtr, Xte = matrices[seed]
    tr, te = splits[seed]
    for w in WEIGHTS:
        weighted = LogisticRegression(max_iter=1000, class_weight={0: 1.0, 1: w})
        model, cpu = fit_and_time(weighted, Xtr, labels[tr], repeats=2)
        weight_rows[w]["iter"].append(int(model.n_iter_[0]))
        weight_rows[w]["time"].append(cpu)
        weight_rows[w]["f1"].append(f1_score(labels[te], model.predict(Xte)))

print(f"{'вага класу «помилка»':>21}{'ітерації':>22}{'у середньому':>14}"
      f"{'проц. час, с':>14}{'F1':>10}")
for w in WEIGHTS:
    steps = np.array(weight_rows[w]["iter"])
    times = np.array(weight_rows[w]["time"])
    scores = np.array(weight_rows[w]["f1"])
    print(f"{w:>21.4f}{str(steps.tolist()):>22}{steps.mean():>14.1f}"
          f"{times.mean():>14.3f}{scores.mean():>10.4f}")
print()
print("Ітерації скачуть від зерна до зерна — це не рівний спуск, а тенденція.")
print("Але сама тенденція стійка: більша вага меншого класу → менше кроків.")

## 15 · А чи дає `balanced` щось, крім швидкості

Тепер найнеприємніше питання до попереднього розділу. `balanced` виграв +0.02 F1 у
звичайної логістичної регресії. Але звичайна логістична регресія приймала рішення
за порогом 0.5, якого ніхто не добирав. А що, як увесь виграш — це просто інший
поріг?

Перевіряємо так само чесно, як зсув у Баєса: поріг добираємо на відкладеній
частині навчання, а число називаємо на перевірній.

In [ ]:
THRESHOLDS = np.round(np.arange(-4.0, 4.01, 0.1), 2)
plain_at_half, plain_tuned, balanced_plain = [], [], []

for seed in (0, 1, 2):
    tr, te = splits[seed]
    fit_idx, dev_idx = train_test_split(tr, test_size=0.2, random_state=seed)
    vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
    Xfit = vec.fit_transform([documents[i] for i in fit_idx])
    Xdev = vec.transform([documents[i] for i in dev_idx])
    Xte = vec.transform([documents[i] for i in te])

    plain = LogisticRegression(max_iter=1000).fit(Xfit, labels[fit_idx])
    dev_score = plain.decision_function(Xdev)
    test_score = plain.decision_function(Xte)
    on_dev = [f1_score(labels[dev_idx], (dev_score + t > 0).astype(int)) for t in THRESHOLDS]
    best_t = float(THRESHOLDS[int(np.argmax(on_dev))])

    balanced = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xfit, labels[fit_idx])

    plain_at_half.append(f1_score(labels[te], (test_score > 0).astype(int)))
    plain_tuned.append(f1_score(labels[te], (test_score + best_t > 0).astype(int)))
    balanced_plain.append(f1_score(labels[te], balanced.predict(Xte)))
    print(f"зерно {seed}: дібраний поріг {best_t:+.2f}"
          f" | звичайна при 0.5 {plain_at_half[-1]:.4f}"
          f" | звичайна з порогом {plain_tuned[-1]:.4f}"
          f" | balanced {balanced_plain[-1]:.4f}")

a, b, c = np.array(plain_at_half), np.array(plain_tuned), np.array(balanced_plain)
print()
print(f"звичайна, поріг 0.5      : {a.mean():.4f} ±{a.std():.4f}")
print(f"звичайна, дібраний поріг : {b.mean():.4f} ±{b.std():.4f}")
print(f"class_weight='balanced'  : {c.mean():.4f} ±{c.std():.4f}")
print(f"balanced − звичайна з порогом: {c.mean() - b.mean():+.4f}"
      f"  при розкиді {max(b.std(), c.std()):.4f}")
print()
print("Якщо ця різниця менша за розкид — уся якісна перевага balanced була порогом,")
print("а справжній його виграш лежить у кількості ітерацій.")

## 16 · Якість і час від розміру навчальної вибірки

Наївний Баєс програє логістичній регресії помітно, але вчиться на порядок швидше.
Це не «поганий метод», а інша точка компромісу — і корисно бачити, як обидві
величини поводяться при різних розмірах даних.

In [ ]:
train_total = len(splits[0][0])
SIZES = sorted({n for n in (2000, 5000, 20000, train_total) if n <= train_total})
curve = collections.defaultdict(lambda: collections.defaultdict(list))

for seed in (0, 1, 2):
    tr, te = splits[seed]
    rng = np.random.default_rng(seed)
    order = rng.permutation(tr)
    for n in SIZES:
        sub = order[:n]
        vec = TfidfVectorizer(lowercase=True, token_pattern=TOKEN_PATTERN, min_df=2)
        Xsub = vec.fit_transform([documents[i] for i in sub])
        Xte = vec.transform([documents[i] for i in te])
        for name, prototype in MODELS:
            model, cpu = fit_and_time(prototype, Xsub, labels[sub], repeats=2)
            curve[n][name + "|f1"].append(f1_score(labels[te], model.predict(Xte)))
            curve[n][name + "|t"].append(cpu)

header = f"{'документів':>11}"
for name, _ in MODELS:
    header += f"{name:>18}"
print(header)
for n in SIZES:
    row = f"{n:>11}"
    for name, _ in MODELS:
        row += f"{np.mean(curve[n][name + '|f1']):>11.4f}/{np.mean(curve[n][name + '|t']):.3f}с"
    print(row)
print()
biggest = SIZES[-1]
smallest = SIZES[0]
print(f"«balanced» проти звичайної логістичної на {smallest} документах: "
      f"{np.mean(curve[smallest]['LogReg balanced|f1']) - np.mean(curve[smallest]['LogReg|f1']):+.4f}")
print(f"те саме на {biggest} документах: "
      f"{np.mean(curve[biggest]['LogReg balanced|f1']) - np.mean(curve[biggest]['LogReg|f1']):+.4f}")
print("Зважування рятує тим більше, чим менше даних.")

## 17 · Що саме вивчила модель

Найдешевша перевірка на ярлик у даних: подивитись на ваги. Якщо серед головних
слів стоять службові дрібнички або назви програм — модель знайшла обхідний шлях,
а не вивчила задачу.

In [ ]:
vec0, Xtr0, Xte0 = matrices[0]
tr0, te0 = splits[0]
best_model = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr0, labels[tr0])
weights = best_model.coef_[0]
names = vec0.get_feature_names_out()
order = np.argsort(weights)

print("НА КОРИСТЬ «ПОМИЛКА» — п'ятнадцять найважчих слів:")
for i in order[::-1][:15]:
    print(f"  {names[i]:<18}{weights[i]:+8.3f}")
print()
print("НА КОРИСТЬ «НЕ ПОМИЛКА»:")
for i in order[:15]:
    print(f"  {names[i]:<18}{weights[i]:+8.3f}")
print()
print(f"ознак усього: {len(names)}")
print(f"найбільша вага за модулем: {np.abs(weights).max():.3f}")
print(f"частка ознак із вагою |w| < 0.5: {(np.abs(weights) < 0.5).mean() * 100:.1f} %")

Слова осмислені — це переклади тих самих понять, з яких зроблено мітку, тільки
українською, і модель дійшла до них сама. Але корисно перевірити й зворотне:
чи не зашилася в модель назва програми. Подивимось, які слова з великою вагою
трапляються лише в одній-двох програмах.

In [ ]:
programs_of_word = collections.defaultdict(set)
tokenize = re.compile(TOKEN_PATTERN).findall
for program, source, target in corpus:
    for token in set(tokenize(target.lower())):
        programs_of_word[token].add(program)

heavy = order[::-1][:60]
narrow = [(names[i], weights[i], len(programs_of_word.get(names[i], ())))
          for i in heavy if len(programs_of_word.get(names[i], ())) <= 2]
print(f"серед шістдесяти найважчих слів «помилки» вузькопрограмних: {len(narrow)}")
for word, w, n_progs in narrow[:10]:
    print(f"  {word:<18}{w:+8.3f}  трапляється у {n_progs} програмах")
if not narrow:
    print("  жодного — усі важкі слова живуть у багатьох програмах одразу")

## 18 · Чи відтворюється це взагалі

Три зерна показують розкид **по даних**. Вони нічого не кажуть про те, чи дає
сама бібліотека однаковий результат на однакових даних. Перевіряємо окремо:
два навчання підряд, ті самі дані, побітове порівняння.

In [ ]:
first = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr0, labels[tr0])
second = LogisticRegression(max_iter=1000, class_weight='balanced').fit(Xtr0, labels[tr0])
identical = np.array_equal(first.coef_, second.coef_)
print("логістична регресія, два прогони побітово однакові:", identical)
assert identical, "розвʼязувач lbfgs мав би бути детермінованим"

nb_first = ComplementNB().fit(Xtr0, labels[tr0]).feature_log_prob_
nb_second = ComplementNB().fit(Xtr0, labels[tr0]).feature_log_prob_
print("ComplementNB, два прогони побітово однакові:", np.array_equal(nb_first, nb_second))
assert np.array_equal(nb_first, nb_second)
print("✅ обидві моделі відтворюються точно")

## 19 · Чому ми весь час дивились на F1, а не на точність

Останній замір теми — і водночас перше питання наступної. Порахуємо, що дає
модель, яка взагалі нічого не вміє: завжди відповідає «не помилка».

In [ ]:
tr, te = splits[0]
y_te = labels[te]
always_zero = np.zeros_like(y_te)
smart = predictions[(0, "LogReg balanced")]

print(f"{'модель':<24}{'точність':>12}{'F1 «помилка»':>16}")
print(f"{'«усе — не помилка»':<24}{accuracy_score(y_te, always_zero):>12.4f}"
      f"{f1_score(y_te, always_zero, zero_division=0):>16.4f}")
print(f"{'LogReg balanced':<24}{accuracy_score(y_te, smart):>12.4f}{f1_score(y_te, smart):>16.4f}")
print()
print(f"точність відрізняє їх на {accuracy_score(y_te, smart) - accuracy_score(y_te, always_zero):.4f},")
print(f"F1 — на {f1_score(y_te, smart) - f1_score(y_te, always_zero, zero_division=0):.4f}.")
print()
print("Дурна модель має точність вище за три чверті просто тому, що клас «помилка»")
print("рідкісний. Що з цим робити — тема 07.")

## Завдання

### 🟢 Рівень 1

Заміни `min_df=2` на `min_df=1` і на `min_df=5` і перерахуй головну таблицю на
трьох зернах. Скільки ознак у кожному випадку і що стається з F1 логістичної
регресії? **Зроблено, якщо** ти назвав кількість ознак для всіх трьох значень і
сказав, чи більша різниця у F1 за розкид по зернах.

### 🟡 Рівень 2

Постав `ngram_range=(1, 2)` — тобто додай до окремих слів ще й пари сусідніх.
Заміряй F1 і процесорний час навчання для всіх чотирьох моделей.
**Зроблено, якщо** в тебе є таблиця «ознак / F1 / час» для (1,1) і (1,2) і
висновок, чи вартий виграш у F1 подорожчання.

### 🔴 Рівень 3

Наївний Баєс на наших даних програє логістичній регресії. Але в розділі 12 ми
побачили, що один зсув додає йому +0.04. Побудуй криву «F1 наївного Баєса від
зсуву» для трьох різних часток класу «помилка» (0.05, 0.2123 і 0.40) і покажи, що
оптимальний зсув рухається разом із часткою. **Зроблено, якщо** ти назвав
оптимальний зсув для кожної з трьох часток і пояснив, чому він рухається саме в
цей бік.